# 11 — CWRU Dataset Inspection and Supplementary Classification Study

This notebook documents the selected Case Western Reserve University (CWRU) 48 kHz drive-end study. It begins with MATLAB-file and signal inspection, then continues through frequency-domain and envelope analysis, dataset-manifest checks, signal segmentation, feature extraction, feature integration, Logistic Regression evaluation and feature-group ablation.

## Selected subset

- Four selected condition series: Normal, IR007, B007 and OR007@6.
- Four motor loads: 0, 1, 2 and 3 HP.
- 16 selected MATLAB recordings.
- 140 complete non-overlapping one-second windows.
- 37 engineered features per window: 9 time-domain, 13 frequency-domain and 15 envelope-domain features.

## Evaluation limitation

The leave-one-load-out and recording-grouped evaluations examine behaviour across the selected loads and files. Different load recordings can still belong to the same selected fault-condition series, so these splits do not establish generalisation to independent physical bearing specimens. Treat this as a supplementary subset study.

In [ ]:

# ============================================================
# STEP 1: CWRU MATLAB FILE INSPECTION
# ============================================================

from pathlib import Path
import numpy as np
from scipy.io import loadmat, whosmat

# ------------------------------------------------------------
# 1. Dataset location
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "cwru_48k_de"
)

EXPECTED_FILES = [
    "Normal_0.mat",
    "IR007_0.mat",
    "B007_0.mat",
    "OR007@6_0.mat"
]

# ------------------------------------------------------------
# 2. Verify files
# ------------------------------------------------------------

print("=" * 70)
print("CWRU DATASET — FILE VERIFICATION")
print("=" * 70)

if not DATA_DIR.is_dir():
    raise FileNotFoundError(
        f"Dataset directory not found: {DATA_DIR}"
    )

missing_files = [
    filename
    for filename in EXPECTED_FILES
    if not (DATA_DIR / filename).is_file()
]

if missing_files:
    raise FileNotFoundError(
        f"Missing files: {missing_files}"
    )

for filename in EXPECTED_FILES:
    file_path = DATA_DIR / filename

    print(
        f"{filename:<20} "
        f"{file_path.stat().st_size / 1024:.2f} KB"
    )

print("\nAll four files are present.")

# ------------------------------------------------------------
# 3. Inspect MATLAB variables and numeric arrays
# ------------------------------------------------------------

inspection_summary = []

for filename in EXPECTED_FILES:

    file_path = DATA_DIR / filename

    print("\n" + "=" * 70)
    print(f"FILE: {filename}")
    print("=" * 70)

    # List variables stored in the MATLAB file
    variable_info = whosmat(file_path)

    print("\nMATLAB variable listing:")
    print("-" * 70)

    for variable_name, shape, matlab_type in variable_info:

        print(
            f"Name: {variable_name:<25} "
            f"Shape: {str(shape):<20} "
            f"MATLAB type: {matlab_type}"
        )

    # Load the file for array inspection
    mat_data = loadmat(
        file_path,
        squeeze_me=True,
        struct_as_record=False
    )

    print("\nNumeric array inspection:")
    print("-" * 70)

    for variable_name, value in mat_data.items():

        # Ignore MATLAB internal metadata
        if variable_name.startswith("__"):
            continue

        array = np.asarray(value)

        # Inspect numeric variables only
        if not np.issubdtype(array.dtype, np.number):
            continue

        if array.size == 0:
            continue

        finite_values = array[np.isfinite(array)]

        print(f"\nVariable: {variable_name}")
        print("  Shape:", array.shape)
        print("  Dimensions:", array.ndim)
        print("  Data type:", array.dtype)
        print("  Total samples/elements:", array.size)
        print(
            "  Finite elements:",
            finite_values.size,
            "/",
            array.size
        )

        if finite_values.size > 0:

            print(
                "  Minimum:",
                float(np.min(finite_values))
            )

            print(
                "  Maximum:",
                float(np.max(finite_values))
            )

            print(
                "  Mean:",
                float(np.mean(finite_values))
            )

            print(
                "  Standard deviation:",
                float(np.std(finite_values))
            )

        # Highlight likely drive-end and RPM variables
        variable_upper = variable_name.upper()

        if "DE_TIME" in variable_upper:
            print("  NOTE: Potential drive-end vibration signal.")

        if "RPM" in variable_upper:
            print("  NOTE: Potential motor speed variable.")

        # Save compact inspection information
        inspection_summary.append({
            "file": filename,
            "variable": variable_name,
            "shape": str(array.shape),
            "dtype": str(array.dtype),
            "elements": int(array.size),
            "finite_elements": int(finite_values.size)
        })

# ------------------------------------------------------------
# 4. Compact inspection summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("INSPECTION SUMMARY")
print("=" * 70)

for item in inspection_summary:
    print(
        f"{item['file']:<18} "
        f"{item['variable']:<25} "
        f"Shape: {item['shape']:<15} "
        f"Elements: {item['elements']}"
    )

print("\nCWRU MATLAB inspection completed.")


## Step 2: Drive-End Signal Extraction and Visualisation

Objective:
- Extract drive-end vibration signals from all four MATLAB files.
- Extract motor RPM.
- Calculate recording duration.
- Verify signal statistics.
- Visualise a short waveform segment.

No filtering, resampling or feature extraction is performed.

In [ ]:

# ============================================================
# STEP 2: CWRU 48 kHz DRIVE-END SIGNAL EXTRACTION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.io import loadmat

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "cwru_48k_de"
)

FS = 48000  # Sampling frequency in Hz

FILE_LABELS = {
    "Normal_0.mat": "Healthy",
    "IR007_0.mat": "Inner",
    "B007_0.mat": "Ball",
    "OR007@6_0.mat": "Outer"
}

# Separate output directory for 48 kHz results.
OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "cwru_inspection_48k"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Store extracted signals and summary information.
signals = {}
summary_rows = []

# ------------------------------------------------------------
# 2. Extract drive-end signals and RPM
# ------------------------------------------------------------

for filename, fault_class in FILE_LABELS.items():

    file_path = DATA_DIR / filename

    if not file_path.is_file():
        raise FileNotFoundError(
            f"File not found: {file_path}"
        )

    mat_data = loadmat(
        file_path,
        squeeze_me=True,
        struct_as_record=False
    )

    # Identify drive-end vibration variable.
    de_keys = [
        key for key in mat_data
        if not key.startswith("__")
        and key.endswith("_DE_time")
    ]

    # Identify RPM variable.
    rpm_keys = [
        key for key in mat_data
        if not key.startswith("__")
        and key.endswith("RPM")
    ]

    if len(de_keys) != 1:
        raise ValueError(
            f"Expected one DE variable in {filename}, "
            f"found: {de_keys}"
        )

    if len(rpm_keys) != 1:
        raise ValueError(
            f"Expected one RPM variable in {filename}, "
            f"found: {rpm_keys}"
        )

    de_key = de_keys[0]
    rpm_key = rpm_keys[0]

    # Extract one-dimensional vibration signal.
    vibration = np.asarray(
        mat_data[de_key],
        dtype=np.float64
    ).reshape(-1)

    rpm = float(
        np.asarray(mat_data[rpm_key]).squeeze()
    )

    # Validate signal.
    if vibration.size == 0:
        raise ValueError(
            f"Empty signal: {filename}"
        )

    if not np.isfinite(vibration).all():
        raise ValueError(
            f"Non-finite values found in {filename}"
        )

    # Calculate recording duration.
    duration = vibration.size / FS

    # Store extracted information.
    signals[fault_class] = {
        "filename": filename,
        "de_variable": de_key,
        "signal": vibration,
        "rpm": rpm,
        "sampling_rate": FS,
        "duration": duration
    }

    # Calculate signal statistics.
    summary_rows.append({
        "Fault Class": fault_class,
        "Filename": filename,
        "DE Variable": de_key,
        "Samples": vibration.size,
        "Sampling Rate (Hz)": FS,
        "Duration (seconds)": duration,
        "RPM": rpm,
        "Mean": np.mean(vibration),
        "Standard Deviation": np.std(vibration),
        "RMS": np.sqrt(np.mean(vibration ** 2)),
        "Minimum": np.min(vibration),
        "Maximum": np.max(vibration)
    })

    print(
        f"Extracted {fault_class:<7} | "
        f"{vibration.size:>7} samples | "
        f"{duration:.3f} seconds | "
        f"{rpm:.0f} RPM"
    )

# ------------------------------------------------------------
# 3. Display signal summary
# ------------------------------------------------------------

summary_df = pd.DataFrame(summary_rows)

print("\n" + "=" * 80)
print("CWRU 48 kHz DRIVE-END SIGNAL SUMMARY")
print("=" * 80)

print(
    summary_df.to_string(
        index=False,
        formatters={
            "Duration (seconds)": "{:.3f}".format,
            "Mean": "{:.6f}".format,
            "Standard Deviation": "{:.6f}".format,
            "RMS": "{:.6f}".format,
            "Minimum": "{:.6f}".format,
            "Maximum": "{:.6f}".format
        }
    )
)

# ------------------------------------------------------------
# 4. Plot first 0.1 seconds of each signal
# ------------------------------------------------------------

plot_duration = 0.1

number_of_samples = int(
    plot_duration * FS
)

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 8)
)

axes = axes.flatten()

for ax, fault_class in zip(
    axes,
    ["Healthy", "Inner", "Ball", "Outer"]
):

    signal = signals[fault_class]["signal"]

    segment = signal[:number_of_samples]

    time_axis = (
        np.arange(len(segment)) / FS
    )

    ax.plot(time_axis, segment)

    ax.set_title(
        f"{fault_class} — "
        f"{signals[fault_class]['rpm']:.0f} RPM"
    )

    ax.set_xlabel("Time (seconds)")
    ax.set_ylabel("Acceleration amplitude")

    ax.grid(True)

fig.suptitle(
    "CWRU 48 kHz Drive-End Vibration Signals — First 0.1 Seconds",
    fontsize=14
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. Save inspection summary
# ------------------------------------------------------------

summary_path = (
    OUTPUT_DIR / "cwru_signal_summary_48k.csv"
)

summary_df.to_csv(
    summary_path,
    index=False
)

print("\nSignal extraction and visualisation completed.")

print("Sampling frequency:", FS, "Hz")
print("Nyquist frequency:", FS / 2, "Hz")
print("Saved summary:", summary_path)


## Step 3: 48 kHz Frequency-Domain Inspection

Objective:
- Calculate the Power Spectral Density using Welch's method.
- Compare frequency distributions across all four conditions.
- Identify dominant spectral peaks.
- Inspect the available frequency range up to the Nyquist frequency.

Sampling frequency: 48,000 Hz
Nyquist frequency: 24,000 Hz

No filtering or envelope feature extraction is performed.

In [ ]:

# ============================================================
# STEP 3: CWRU 48 kHz FREQUENCY-DOMAIN INSPECTION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from scipy.signal import welch

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "cwru_48k_de"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "cwru_inspection_48k"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FS = 48000
NYQUIST = FS / 2

FILE_LABELS = {
    "Normal_0.mat": "Healthy",
    "IR007_0.mat": "Inner",
    "B007_0.mat": "Ball",
    "OR007@6_0.mat": "Outer"
}

# Welch PSD parameters
N_PER_SEG = 16384
N_OVERLAP = 8192

# ------------------------------------------------------------
# 2. Load signals and calculate PSD
# ------------------------------------------------------------

spectra = {}
spectral_rows = []

for filename, fault_class in FILE_LABELS.items():

    file_path = DATA_DIR / filename

    if not file_path.is_file():
        raise FileNotFoundError(file_path)

    mat_data = loadmat(
        file_path,
        squeeze_me=True,
        struct_as_record=False
    )

    de_keys = [
        key for key in mat_data
        if not key.startswith("__")
        and key.endswith("_DE_time")
    ]

    if len(de_keys) != 1:
        raise ValueError(
            f"Could not uniquely identify DE signal "
            f"in {filename}: {de_keys}"
        )

    signal = np.asarray(
        mat_data[de_keys[0]],
        dtype=np.float64
    ).reshape(-1)

    if not np.isfinite(signal).all():
        raise ValueError(
            f"Non-finite values found in {filename}"
        )

    # --------------------------------------------------------
    # Welch Power Spectral Density
    # --------------------------------------------------------

    frequencies, psd = welch(
        signal,
        fs=FS,
        window="hann",
        nperseg=N_PER_SEG,
        noverlap=N_OVERLAP,
        detrend="constant",
        scaling="density"
    )

    # Exclude DC and near-DC components from peak search.
    valid_mask = (
        (frequencies >= 10) &
        (frequencies <= NYQUIST)
    )

    valid_indices = np.flatnonzero(valid_mask)

    if len(valid_indices) == 0:
        raise ValueError(
            f"No valid frequency bins in {filename}"
        )

    peak_index = valid_indices[
        np.argmax(psd[valid_mask])
    ]

    peak_frequency = frequencies[peak_index]
    peak_psd = psd[peak_index]

    # Store spectrum.
    spectra[fault_class] = {
        "frequencies": frequencies,
        "psd": psd,
        "peak_frequency": peak_frequency,
        "peak_psd": peak_psd
    }

    spectral_rows.append({
        "Fault Class": fault_class,
        "Filename": filename,
        "Peak Frequency (Hz)": peak_frequency,
        "Peak PSD": peak_psd,
        "Frequency Resolution (Hz)": FS / N_PER_SEG
    })

    print(
        f"{fault_class:<8} | "
        f"Peak: {peak_frequency:9.2f} Hz | "
        f"PSD: {peak_psd:.6e}"
    )

# ------------------------------------------------------------
# 3. Create spectral summary
# ------------------------------------------------------------

spectral_summary = pd.DataFrame(spectral_rows)

print("\n" + "=" * 75)
print("CWRU 48 kHz SPECTRAL SUMMARY")
print("=" * 75)

print(
    spectral_summary.to_string(
        index=False,
        formatters={
            "Peak Frequency (Hz)": "{:.2f}".format,
            "Peak PSD": "{:.6e}".format,
            "Frequency Resolution (Hz)": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# 4. Plot all four PSD curves
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 9),
    sharex=True,
    sharey=True
)

axes = axes.flatten()

for ax, fault_class in zip(
    axes,
    ["Healthy", "Inner", "Ball", "Outer"]
):

    frequencies = spectra[fault_class]["frequencies"]
    psd = spectra[fault_class]["psd"]

    psd_db = 10 * np.log10(
        np.maximum(psd, 1e-18)
    )

    ax.plot(frequencies, psd_db)

    peak_frequency = (
        spectra[fault_class]["peak_frequency"]
    )

    ax.axvline(
        peak_frequency,
        linestyle="--",
        label=f"Peak: {peak_frequency:.2f} Hz"
    )

    ax.set_title(
        f"{fault_class} — Drive-End PSD"
    )

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("PSD (dB/Hz)")

    ax.set_xlim(0, NYQUIST)
    ax.grid(True)
    ax.legend()

fig.suptitle(
    "CWRU 48 kHz Drive-End Power Spectral Density",
    fontsize=14
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 5. Save results
# ------------------------------------------------------------

summary_path = (
    OUTPUT_DIR
    / "cwru_spectral_summary_48k.csv"
)

spectral_summary.to_csv(
    summary_path,
    index=False
)

print("\n48 kHz frequency-domain inspection completed.")

print("Sampling frequency:", FS, "Hz")
print("Nyquist frequency:", NYQUIST, "Hz")
print(
    "Frequency resolution:",
    FS / N_PER_SEG,
    "Hz"
)

print("Saved:", summary_path)


## Step 4: Theoretical Bearing Fault Frequency Calculation

Objective:
- Identify the CWRU drive-end bearing geometry.
- Calculate shaft rotational frequency.
- Calculate characteristic bearing defect frequencies.
- Calculate harmonics for subsequent envelope-spectrum analysis.

Bearing: SKF 6205-2RS JEM
Sampling frequency: 48 kHz
Measured motor speed: 1796 RPM

In [ ]:

# ============================================================
# STEP 4: CWRU THEORETICAL BEARING FAULT FREQUENCIES
# Includes conventional BSF and 2xBSF
# ============================================================

from pathlib import Path
import pandas as pd

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "cwru_inspection_48k"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RPM = 1796.0
SHAFT_FREQUENCY = RPM / 60.0

# CWRU SKF 6205-2RS JEM frequency multipliers.
FAULT_MULTIPLIERS = {
    "FTF - Cage": 0.39828,
    "BPFO - Outer Race": 3.5848,
    "BPFI - Inner Race": 5.4152,
    "BSF - Ball Spin": 4.7135 / 2,
    "2xBSF - Ball Impact": 4.7135
}

# ------------------------------------------------------------
# 2. Calculate fundamental frequencies and harmonics
# ------------------------------------------------------------

rows = []

for characteristic, multiplier in FAULT_MULTIPLIERS.items():

    fundamental = multiplier * SHAFT_FREQUENCY

    row = {
        "Characteristic": characteristic,
        "Multiplier": multiplier,
        "Fundamental_Hz": fundamental
    }

    for harmonic in range(2, 6):
        row[f"{harmonic}x_Hz"] = (
            fundamental * harmonic
        )

    rows.append(row)

frequency_df = pd.DataFrame(rows)

# ------------------------------------------------------------
# 3. Display results
# ------------------------------------------------------------

print("=" * 70)
print("CWRU BEARING FREQUENCY REFERENCE")
print("=" * 70)

print(f"Measured RPM: {RPM:.0f}")
print(f"Shaft frequency: {SHAFT_FREQUENCY:.4f} Hz")

print("\n" + "=" * 95)
print("CHARACTERISTIC FREQUENCIES AND HARMONICS")
print("=" * 95)

print(
    frequency_df.to_string(
        index=False,
        formatters={
            "Multiplier": "{:.5f}".format,
            "Fundamental_Hz": "{:.3f}".format,
            "2x_Hz": "{:.3f}".format,
            "3x_Hz": "{:.3f}".format,
            "4x_Hz": "{:.3f}".format,
            "5x_Hz": "{:.3f}".format
        }
    )
)

# ------------------------------------------------------------
# 4. Save frequency reference
# ------------------------------------------------------------

OUTPUT_PATH = (
    OUTPUT_DIR
    / "cwru_bearing_fault_frequencies_48k.csv"
)

frequency_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print("\nFrequency calculation completed.")
print("Saved:", OUTPUT_PATH)


## Step 5: CWRU Envelope Analysis

Objective:
- Apply Butterworth bandpass filtering to the drive-end signals.
- Extract amplitude envelopes using the Hilbert transform.
- Calculate envelope Power Spectral Density using Welch's method.
- Inspect characteristic bearing frequencies and their harmonics.

Configuration:
- Sampling frequency: 48 kHz
- Bandpass: 500–5000 Hz
- Filter order: 4
- Envelope spectrum range: 0–1000 Hz

This is an exploratory signal-processing experiment.

In [ ]:

# ============================================================
# STEP 5: CWRU BANDPASS FILTERING AND ENVELOPE ANALYSIS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scipy.io import loadmat
from scipy.signal import butter, sosfiltfilt, hilbert, welch

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "cwru_48k_de"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "cwru_inspection_48k"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FS = 48000
NYQUIST = FS / 2

LOWCUT = 500
HIGHCUT = 5000
FILTER_ORDER = 4

MAX_ENVELOPE_FREQUENCY = 1000

# Welch parameters for envelope spectrum.
N_PER_SEG = 65536
N_OVERLAP = 32768

FILE_LABELS = {
    "Normal_0.mat": "Healthy",
    "IR007_0.mat": "Inner",
    "B007_0.mat": "Ball",
    "OR007@6_0.mat": "Outer"
}

# Standard CWRU SKF 6205-2RS JEM multipliers.
FAULT_MULTIPLIERS = {
    "Inner": 5.4152,       # BPFI
    "Ball": 2.35675,       # BSF
    "Outer": 3.5848        # BPFO
}

HARMONICS = range(1, 6)
PEAK_TOLERANCE_HZ = 4.0

# ------------------------------------------------------------
# 2. Validate filter configuration
# ------------------------------------------------------------

if not (0 < LOWCUT < HIGHCUT < NYQUIST):
    raise ValueError(
        "Invalid bandpass frequencies for the sampling rate."
    )

if N_PER_SEG > 243000:
    raise ValueError(
        "Welch segment length is too large for the recordings."
    )

print("=" * 70)
print("CWRU ENVELOPE ANALYSIS CONFIGURATION")
print("=" * 70)

print("Sampling frequency:", FS, "Hz")
print("Nyquist frequency:", NYQUIST, "Hz")
print("Bandpass:", LOWCUT, "-", HIGHCUT, "Hz")
print("Filter order:", FILTER_ORDER)
print("Envelope spectrum range:", MAX_ENVELOPE_FREQUENCY, "Hz")
print("Welch frequency resolution:", FS / N_PER_SEG, "Hz")

# ------------------------------------------------------------
# 3. Design Butterworth bandpass filter
# ------------------------------------------------------------

sos = butter(
    FILTER_ORDER,
    [LOWCUT, HIGHCUT],
    btype="bandpass",
    fs=FS,
    output="sos"
)

# ------------------------------------------------------------
# 4. Process each recording
# ------------------------------------------------------------

envelope_results = {}
harmonic_rows = []
global_peak_rows = []

for filename, fault_class in FILE_LABELS.items():

    file_path = DATA_DIR / filename

    if not file_path.is_file():
        raise FileNotFoundError(file_path)

    mat_data = loadmat(
        file_path,
        squeeze_me=True,
        struct_as_record=False
    )

    # Identify DE and RPM variables.
    de_keys = [
        key for key in mat_data
        if not key.startswith("__")
        and key.endswith("_DE_time")
    ]

    rpm_keys = [
        key for key in mat_data
        if not key.startswith("__")
        and key.endswith("RPM")
    ]

    if len(de_keys) != 1 or len(rpm_keys) != 1:
        raise ValueError(
            f"Unexpected MATLAB structure in {filename}"
        )

    signal = np.asarray(
        mat_data[de_keys[0]],
        dtype=np.float64
    ).reshape(-1)

    rpm = float(
        np.asarray(mat_data[rpm_keys[0]]).squeeze()
    )

    if not np.isfinite(signal).all():
        raise ValueError(
            f"Non-finite values in {filename}"
        )

    # --------------------------------------------------------
    # 4A. Bandpass filtering
    # --------------------------------------------------------

    filtered_signal = sosfiltfilt(
        sos,
        signal
    )

    # --------------------------------------------------------
    # 4B. Extract amplitude envelope
    # --------------------------------------------------------

    analytic_signal = hilbert(
        filtered_signal
    )

    envelope = np.abs(
        analytic_signal
    )

    # Remove DC component from envelope.
    envelope_demeaned = envelope - np.mean(envelope)

    # --------------------------------------------------------
    # 4C. Calculate envelope PSD
    # --------------------------------------------------------

    frequencies, envelope_psd = welch(
        envelope_demeaned,
        fs=FS,
        window="hann",
        nperseg=N_PER_SEG,
        noverlap=N_OVERLAP,
        detrend="constant",
        scaling="density"
    )

    frequency_mask = (
        (frequencies >= 1) &
        (frequencies <= MAX_ENVELOPE_FREQUENCY)
    )

    valid_frequencies = frequencies[frequency_mask]
    valid_psd = envelope_psd[frequency_mask]

    # Find global peak, excluding near-DC frequencies.
    global_peak_index = np.argmax(valid_psd)

    global_peak_frequency = (
        valid_frequencies[global_peak_index]
    )

    global_peak_value = valid_psd[global_peak_index]

    global_peak_rows.append({
        "Fault Class": fault_class,
        "RPM": rpm,
        "Global Envelope Peak (Hz)": global_peak_frequency,
        "Global Envelope Peak PSD": global_peak_value
    })

    # Save processed data for analysis and plotting.
    envelope_results[fault_class] = {
        "filename": filename,
        "rpm": rpm,
        "original_signal": signal,
        "filtered_signal": filtered_signal,
        "envelope": envelope,
        "frequencies": valid_frequencies,
        "psd": valid_psd
    }

    # --------------------------------------------------------
    # 4D. Compare expected fault harmonics
    # --------------------------------------------------------

    if fault_class in FAULT_MULTIPLIERS:

        shaft_frequency = rpm / 60.0

        fundamental = (
            FAULT_MULTIPLIERS[fault_class]
            * shaft_frequency
        )

        for harmonic in HARMONICS:

            expected_frequency = (
                fundamental * harmonic
            )

            # Find maximum PSD within a tolerance window.
            local_mask = (
                (valid_frequencies >=
                 expected_frequency - PEAK_TOLERANCE_HZ)
                &
                (valid_frequencies <=
                 expected_frequency + PEAK_TOLERANCE_HZ)
            )

            if not np.any(local_mask):
                detected_frequency = np.nan
                detected_psd = np.nan
            else:
                local_freqs = valid_frequencies[local_mask]
                local_values = valid_psd[local_mask]

                best_index = np.argmax(local_values)

                detected_frequency = local_freqs[best_index]
                detected_psd = local_values[best_index]

            harmonic_rows.append({
                "Fault Class": fault_class,
                "Harmonic": harmonic,
                "Expected Frequency (Hz)": expected_frequency,
                "Detected Local Peak (Hz)": detected_frequency,
                "Peak PSD": detected_psd,
                "Frequency Error (Hz)": (
                    detected_frequency - expected_frequency
                    if np.isfinite(detected_frequency)
                    else np.nan
                )
            })

    print(
        f"{fault_class:<8} | "
        f"RPM: {rpm:.0f} | "
        f"Global envelope peak: "
        f"{global_peak_frequency:.2f} Hz"
    )

# ------------------------------------------------------------
# 5. Plot envelope spectra
# ------------------------------------------------------------

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 9),
    sharex=True
)

axes = axes.flatten()

for ax, fault_class in zip(
    axes,
    ["Healthy", "Inner", "Ball", "Outer"]
):

    result = envelope_results[fault_class]

    frequencies = result["frequencies"]
    psd = result["psd"]

    psd_db = 10 * np.log10(
        np.maximum(psd, 1e-18)
    )

    ax.plot(frequencies, psd_db)

    # Mark theoretical harmonics for each fault category.
    if fault_class in FAULT_MULTIPLIERS:

        shaft_frequency = result["rpm"] / 60.0

        fundamental = (
            FAULT_MULTIPLIERS[fault_class]
            * shaft_frequency
        )

        for harmonic in HARMONICS:

            expected_frequency = fundamental * harmonic

            ax.axvline(
                expected_frequency,
                linestyle="--",
                alpha=0.55
            )

        ax.set_title(
            f"{fault_class} — Envelope PSD "
            f"(Fundamental: {fundamental:.2f} Hz)"
        )

    else:
        ax.set_title(
            "Healthy — Envelope PSD"
        )

    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Envelope PSD (dB/Hz)")
    ax.set_xlim(0, MAX_ENVELOPE_FREQUENCY)
    ax.grid(True)

fig.suptitle(
    "CWRU 48 kHz Envelope Spectra — 500–5000 Hz Bandpass",
    fontsize=14
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 6. Display global envelope peak summary
# ------------------------------------------------------------

global_peak_df = pd.DataFrame(global_peak_rows)

print("\n" + "=" * 80)
print("GLOBAL ENVELOPE PEAK SUMMARY")
print("=" * 80)

print(
    global_peak_df.to_string(
        index=False,
        formatters={
            "Global Envelope Peak (Hz)": "{:.3f}".format,
            "Global Envelope Peak PSD": "{:.6e}".format
        }
    )
)

# ------------------------------------------------------------
# 7. Display characteristic harmonic comparison
# ------------------------------------------------------------

harmonic_df = pd.DataFrame(harmonic_rows)

print("\n" + "=" * 95)
print("THEORETICAL VS OBSERVED ENVELOPE HARMONICS")
print("=" * 95)

print(
    harmonic_df.to_string(
        index=False,
        formatters={
            "Expected Frequency (Hz)": "{:.3f}".format,
            "Detected Local Peak (Hz)": "{:.3f}".format,
            "Peak PSD": "{:.6e}".format,
            "Frequency Error (Hz)": "{:.3f}".format
        }
    )
)

# ------------------------------------------------------------
# 8. Save results
# ------------------------------------------------------------

global_peak_path = (
    OUTPUT_DIR / "cwru_envelope_global_peaks_48k.csv"
)

harmonic_path = (
    OUTPUT_DIR / "cwru_envelope_harmonic_comparison_48k.csv"
)

global_peak_df.to_csv(
    global_peak_path,
    index=False
)

harmonic_df.to_csv(
    harmonic_path,
    index=False
)

print("\nEnvelope analysis completed.")

print("Saved global peak summary:", global_peak_path)
print("Saved harmonic comparison:", harmonic_path)


## Step 6: CWRU Dataset Manifest and Quality Verification

Objective:
- Inspect all 16 CWRU recordings.
- Identify fault class and motor load from filenames.
- Extract RPM and drive-end vibration signal.
- Calculate recording duration and statistical properties.
- Verify class and load distribution.
- Generate a consolidated dataset manifest.

No filtering, segmentation or feature extraction is performed.

In [ ]:

# ============================================================
# DIAGNOSTIC: INSPECT NORMAL_1.MAT VARIABLES
# ============================================================

from pathlib import Path
from scipy.io import loadmat, whosmat

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

file_path = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "cwru_48k_de"
    / "Normal_1.mat"
)

print("=" * 65)
print("NORMAL_1.MAT — MATLAB VARIABLE INSPECTION")
print("=" * 65)

# Inspect variable names and dimensions
print("\nMATLAB variable listing:")

for name, shape, matlab_type in whosmat(file_path):
    print(
        f"Name: {name!r:<30} "
        f"Shape: {str(shape):<20} "
        f"Type: {matlab_type}"
    )

# Load and inspect every key
mat_data = loadmat(
    file_path,
    squeeze_me=True,
    struct_as_record=False
)

print("\nAll loaded MATLAB keys:")

for key, value in mat_data.items():

    print("\nKey:", repr(key))

    if not key.startswith("__"):
        print("Type:", type(value))
        print("Shape:", getattr(value, "shape", None))
        print("Value:", value if key.lower().endswith("rpm") else "")


In [ ]:

from pathlib import Path
import numpy as np
from scipy.io import loadmat

DATA_DIR = (
    (Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd())
    / "data" / "raw" / "cwru_48k_de"
)

normal_1 = loadmat(DATA_DIR / "Normal_1.mat", squeeze_me=True)
normal_2 = loadmat(DATA_DIR / "Normal_2.mat", squeeze_me=True)

print("Normal_1 variables:",
      [k for k in normal_1 if not k.startswith("__")])

print("\nNormal_2 variables:",
      [k for k in normal_2 if not k.startswith("__")])

for channel in ["DE", "FE"]:
    key = f"X098_{channel}_time"

    if key in normal_1 and key in normal_2:
        a = np.asarray(normal_1[key]).ravel()
        b = np.asarray(normal_2[key]).ravel()

        print(f"\n{key}")
        print("Same shape:", a.shape == b.shape)
        print("Exactly identical:", np.array_equal(a, b))

        if a.shape == b.shape:
            print("Maximum absolute difference:",
                  np.max(np.abs(a - b)))


In [ ]:

from pathlib import Path
import numpy as np
from scipy.io import loadmat

DATA_DIR = (
    (Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd())
    / "data" / "raw" / "cwru_48k_de"
)

mat = loadmat(
    DATA_DIR / "Normal_2.mat",
    squeeze_me=True
)

signal = np.asarray(
    mat["X099_DE_time"],
    dtype=np.float64
).ravel()

print("Selected signal: X099_DE_time")
print("Samples:", len(signal))
print("Duration at 48 kHz:", len(signal) / 48000)
print("Finite values:", np.isfinite(signal).all())
print("Mean:", np.mean(signal))
print("Standard deviation:", np.std(signal))
print("RMS:", np.sqrt(np.mean(signal ** 2)))
print("Minimum:", np.min(signal))
print("Maximum:", np.max(signal))


In [ ]:

# ============================================================
# STEP 6A: CONFIGURATION AND FILE VERIFICATION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import loadmat

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_DIR = (
    PROJECT_ROOT / "data" / "raw" / "cwru_48k_de"
)

OUTPUT_DIR = (
    PROJECT_ROOT / "results" / "cwru_inspection_48k"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FS = 48000

CLASS_MAPPING = {
    "Normal": "Healthy",
    "IR007": "Inner",
    "B007": "Ball",
    "OR007@6": "Outer"
}

NORMAL_VARIABLES = {
    0: "X097_DE_time",
    1: "X098_DE_time",
    2: "X099_DE_time",
    3: "X100_DE_time"
}

REFERENCE_RPM = {
    0: 1797,
    1: 1772,
    2: 1750,
    3: 1730
}

EXPECTED_FILES = [
    f"{prefix}_{load}.mat"
    for prefix in CLASS_MAPPING
    for load in range(4)
]

# Verify folder and files.
if not DATA_DIR.is_dir():
    raise FileNotFoundError(DATA_DIR)

actual_files = {
    path.name for path in DATA_DIR.glob("*.mat")
}

missing = sorted(set(EXPECTED_FILES) - actual_files)
unexpected = sorted(actual_files - set(EXPECTED_FILES))

print("Expected files:", len(EXPECTED_FILES))
print("Found files:", len(actual_files))
print("Missing:", missing)
print("Unexpected:", unexpected)

if missing or unexpected:
    raise ValueError("Dataset file verification failed.")

print("\nAll 16 expected recordings are present.")
print("Sampling rate:", FS, "Hz")
print("Output directory:", OUTPUT_DIR)


In [ ]:

# ============================================================
# STEP 6B: EXTRACT RECORDING METADATA
# ============================================================

manifest_rows = []

for filename in sorted(EXPECTED_FILES):

    file_path = DATA_DIR / filename

    # Identify class and motor load.
    prefix, load_text = Path(filename).stem.rsplit("_", 1)

    fault_class = CLASS_MAPPING[prefix]
    load_hp = int(load_text)

    reference_rpm = REFERENCE_RPM[load_hp]

    # Load MATLAB file.
    mat_data = loadmat(
        file_path,
        squeeze_me=True,
        struct_as_record=False
    )

    # --------------------------------------------------------
    # Select drive-end signal
    # --------------------------------------------------------

    if fault_class == "Healthy":

        # Explicit selection avoids duplicated X098 in Normal_2.
        de_variable = NORMAL_VARIABLES[load_hp]

        if de_variable not in mat_data:
            raise ValueError(
                f"{de_variable} missing from {filename}"
            )

    else:

        de_keys = [
            key for key in mat_data
            if not key.startswith("__")
            and key.endswith("_DE_time")
        ]

        if len(de_keys) != 1:
            raise ValueError(
                f"Unexpected DE variables in "
                f"{filename}: {de_keys}"
            )

        de_variable = de_keys[0]

    signal = np.asarray(
        mat_data[de_variable],
        dtype=np.float64
    ).reshape(-1)

    # --------------------------------------------------------
    # Extract RPM
    # --------------------------------------------------------

    rpm_keys = [
        key for key in mat_data
        if not key.startswith("__")
        and key.endswith("RPM")
    ]

    if len(rpm_keys) > 1:
        raise ValueError(
            f"Multiple RPM variables in {filename}"
        )

    if len(rpm_keys) == 1:

        measured_rpm = float(
            np.asarray(mat_data[rpm_keys[0]]).squeeze()
        )

        rpm_for_analysis = measured_rpm
        rpm_source = "Measured from MAT file"

    else:

        measured_rpm = np.nan
        rpm_for_analysis = float(reference_rpm)
        rpm_source = "Nominal CWRU reference"

    # --------------------------------------------------------
    # Validate signal
    # --------------------------------------------------------

    if signal.size == 0:
        raise ValueError(f"Empty signal: {filename}")

    if not np.isfinite(signal).all():
        raise ValueError(
            f"Non-finite values: {filename}"
        )

    # --------------------------------------------------------
    # Calculate signal statistics
    # --------------------------------------------------------

    sample_count = len(signal)
    duration = sample_count / FS

    mean_value = float(np.mean(signal))
    std_value = float(np.std(signal))
    rms_value = float(np.sqrt(np.mean(signal ** 2)))

    # --------------------------------------------------------
    # Add one recording to manifest
    # --------------------------------------------------------

    manifest_rows.append({
        "filename": filename,
        "fault_class": fault_class,
        "motor_load_hp": load_hp,
        "reference_rpm": reference_rpm,
        "measured_rpm": measured_rpm,
        "rpm_for_analysis": rpm_for_analysis,
        "rpm_source": rpm_source,
        "sampling_rate_hz": FS,
        "de_variable": de_variable,
        "sample_count": sample_count,
        "duration_seconds": duration,
        "mean": mean_value,
        "std": std_value,
        "rms": rms_value,
        "peak_to_peak": float(np.ptp(signal)),
        "min": float(np.min(signal)),
        "max": float(np.max(signal)),
        "finite_values": int(np.isfinite(signal).sum()),
        "file_size_kb": file_path.stat().st_size / 1024
    })

    print(
        f"{filename:<18} | "
        f"{fault_class:<7} | "
        f"{load_hp} HP | "
        f"{rpm_for_analysis:.0f} RPM | "
        f"{de_variable}"
    )

print("\nRecordings processed:", len(manifest_rows))

assert len(manifest_rows) == 16


In [ ]:

# ============================================================
# STEP 6C: MANIFEST AND DATASET QUALITY CHECKS
# ============================================================

if not manifest_rows:
    raise ValueError(
        "No manifest records found. Run Cell 6B first."
    )

manifest_df = pd.DataFrame(manifest_rows)

required_columns = [
    "filename",
    "fault_class",
    "motor_load_hp",
    "measured_rpm",
    "rpm_for_analysis",
    "sample_count",
    "duration_seconds",
    "rms"
]

missing_columns = [
    col for col in required_columns
    if col not in manifest_df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing manifest columns: {missing_columns}"
    )

manifest_df = manifest_df.sort_values(
    ["motor_load_hp", "fault_class"]
).reset_index(drop=True)

print("=" * 75)
print("CWRU 48 kHz DATASET MANIFEST")
print("=" * 75)

print(
    manifest_df[required_columns].to_string(index=False)
)

# ------------------------------------------------------------
# Class distribution
# ------------------------------------------------------------

print("\nFAULT CLASS DISTRIBUTION")

class_distribution = (
    manifest_df["fault_class"]
    .value_counts()
    .sort_index()
)

print(class_distribution)

# ------------------------------------------------------------
# Motor load distribution
# ------------------------------------------------------------

print("\nMOTOR LOAD DISTRIBUTION")

load_distribution = (
    manifest_df["motor_load_hp"]
    .value_counts()
    .sort_index()
)

print(load_distribution)

# ------------------------------------------------------------
# Class versus load
# ------------------------------------------------------------

print("\nFAULT CLASS BY MOTOR LOAD")

class_load_table = pd.crosstab(
    manifest_df["fault_class"],
    manifest_df["motor_load_hp"]
)

print(class_load_table)

# ------------------------------------------------------------
# RPM summary
# ------------------------------------------------------------

print("\nRPM DATA AVAILABILITY")

print(
    manifest_df["rpm_source"].value_counts()
)

rpm_summary = (
    manifest_df.groupby("motor_load_hp")
    .agg(
        recordings=("filename", "count"),
        measured_rpm_count=("measured_rpm", "count"),
        mean_analysis_rpm=("rpm_for_analysis", "mean"),
        minimum_rpm=("rpm_for_analysis", "min"),
        maximum_rpm=("rpm_for_analysis", "max")
    )
)

print("\nRPM SUMMARY")
print(rpm_summary)


In [ ]:

# ============================================================
# STEP 6D: FINAL VALIDATION AND SAVE
# ============================================================

checks = {
    "16 recordings available":
        len(manifest_df) == 16,

    "Four recordings per fault class":
        (class_distribution == 4).all(),

    "Four recordings per motor load":
        (load_distribution == 4).all(),

    "All classes represented at every load":
        (class_load_table == 1).all().all(),

    "All recordings use 48 kHz":
        (manifest_df["sampling_rate_hz"] == FS).all(),

    "All signals contain finite values":
        (
            manifest_df["finite_values"]
            == manifest_df["sample_count"]
        ).all(),

    "All analysis RPM values are valid":
        (
            manifest_df["rpm_for_analysis"].notna()
            & (manifest_df["rpm_for_analysis"] > 0)
        ).all()
}

print("=" * 70)
print("FINAL DATASET QUALITY CHECK")
print("=" * 70)

for description, passed in checks.items():
    print(
        f"{'PASS' if passed else 'FAIL'}: {description}"
    )

if not all(checks.values()):
    raise ValueError(
        "Dataset verification failed."
    )

# Save manifest.
manifest_path = (
    OUTPUT_DIR / "cwru_48k_dataset_manifest.csv"
)

manifest_df.to_csv(
    manifest_path,
    index=False
)

# Save class/load distribution.
class_load_path = (
    OUTPUT_DIR / "cwru_48k_class_load_distribution.csv"
)

class_load_table.to_csv(
    class_load_path
)

# Save RPM summary.
rpm_path = (
    OUTPUT_DIR / "cwru_48k_rpm_summary.csv"
)

rpm_summary.to_csv(rpm_path)

print("\nAll checks passed.")
print("Manifest saved:", manifest_path)
print("Class/load table saved:", class_load_path)
print("RPM summary saved:", rpm_path)


## Step 7: Signal Segmentation Planning

### Step 7A: Window Count Audit

Calculate the number of complete non-overlapping
1-second windows available from each recording.

No signal modification is performed.

In [ ]:

# ============================================================
# STEP 7A: WINDOW COUNT AUDIT
# ============================================================

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

MANIFEST_PATH = (
    PROJECT_ROOT
    / "results"
    / "cwru_inspection_48k"
    / "cwru_48k_dataset_manifest.csv"
)

manifest = pd.read_csv(MANIFEST_PATH)

FS = 48000
WINDOW_SECONDS = 1.0
WINDOW_SAMPLES = int(FS * WINDOW_SECONDS)

window_plan = manifest.copy()

window_plan["window_samples"] = WINDOW_SAMPLES

window_plan["complete_windows"] = (
    window_plan["sample_count"] // WINDOW_SAMPLES
).astype(int)

window_plan["discarded_samples"] = (
    window_plan["sample_count"] % WINDOW_SAMPLES
).astype(int)

window_plan["discarded_duration_seconds"] = (
    window_plan["discarded_samples"] / FS
)

print("Window duration:", WINDOW_SECONDS, "seconds")
print("Samples per window:", WINDOW_SAMPLES)

print("\nRECORDING-WISE WINDOW PLAN")

print(
    window_plan[
        [
            "filename",
            "fault_class",
            "motor_load_hp",
            "sample_count",
            "complete_windows",
            "discarded_samples"
        ]
    ].to_string(index=False)
)

print("\nWINDOW COUNTS BY FAULT CLASS AND LOAD")

window_table = pd.pivot_table(
    window_plan,
    values="complete_windows",
    index="fault_class",
    columns="motor_load_hp",
    aggfunc="sum",
    fill_value=0
)

print(window_table)

print("\nTOTAL WINDOWS:", window_plan["complete_windows"].sum())

print(
    "Total discarded samples:",
    window_plan["discarded_samples"].sum()
)

# Save the segmentation plan.
OUTPUT_PATH = (
    PROJECT_ROOT
    / "results"
    / "cwru_inspection_48k"
    / "cwru_48k_window_plan.csv"
)

window_plan.to_csv(OUTPUT_PATH, index=False)

print("\nSaved:", OUTPUT_PATH)


### Step 7B: Raw Signal Segmentation

Divide each original drive-end vibration recording into
complete, non-overlapping one-second windows.

Each window contains 48,000 samples at 48 kHz.

No filtering, normalisation or feature extraction is applied.

In [ ]:

# ============================================================
# STEP 7B: RAW SIGNAL SEGMENTATION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.io import loadmat

# ------------------------------------------------------------
# 1. Configuration and manifest
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "cwru_48k_de"

MANIFEST_PATH = (
    PROJECT_ROOT
    / "results"
    / "cwru_inspection_48k"
    / "cwru_48k_dataset_manifest.csv"
)

FS = 48000
WINDOW_SAMPLES = 48000

LABEL_MAPPING = {
    "Healthy": 0,
    "Inner": 1,
    "Ball": 2,
    "Outer": 3
}

manifest = pd.read_csv(MANIFEST_PATH)

# ------------------------------------------------------------
# 2. Generate windows
# ------------------------------------------------------------

windows = []
window_metadata = []

for _, row in manifest.iterrows():

    filename = row["filename"]
    file_path = DATA_DIR / filename

    mat_data = loadmat(
        file_path,
        squeeze_me=True,
        struct_as_record=False
    )

    # Use the verified variable from the manifest.
    signal = np.asarray(
        mat_data[row["de_variable"]],
        dtype=np.float64
    ).reshape(-1)

    if len(signal) != int(row["sample_count"]):
        raise ValueError(
            f"Sample count mismatch in {filename}"
        )

    if not np.isfinite(signal).all():
        raise ValueError(
            f"Non-finite samples in {filename}"
        )

    # Number of complete one-second windows.
    number_of_windows = (
        len(signal) // WINDOW_SAMPLES
    )

    for window_index in range(number_of_windows):

        start = window_index * WINDOW_SAMPLES
        end = start + WINDOW_SAMPLES

        segment = signal[start:end]

        windows.append(
            segment.astype(np.float32)
        )

        window_metadata.append({
            "filename": filename,
            "fault_class": row["fault_class"],
            "label": LABEL_MAPPING[row["fault_class"]],
            "motor_load_hp": int(row["motor_load_hp"]),
            "rpm": float(row["rpm_for_analysis"]),
            "rpm_source": row["rpm_source"],
            "window_index": window_index + 1,
            "start_sample": start,
            "end_sample": end,
            "start_time_seconds": start / FS,
            "end_time_seconds": end / FS
        })

    print(
        f"{filename:<18} "
        f"| Windows: {number_of_windows:2d} "
        f"| Fault: {row['fault_class']:<7} "
        f"| Load: {int(row['motor_load_hp'])} HP"
    )

# ------------------------------------------------------------
# 3. Convert to arrays and DataFrame
# ------------------------------------------------------------

X_windows = np.stack(windows).astype(np.float32)

window_metadata_df = pd.DataFrame(window_metadata)

# ------------------------------------------------------------
# 4. Basic verification
# ------------------------------------------------------------

assert X_windows.ndim == 2
assert X_windows.shape[1] == WINDOW_SAMPLES
assert X_windows.shape[0] == len(window_metadata_df)
assert np.isfinite(X_windows).all()

print("\n" + "=" * 65)
print("RAW WINDOW DATASET CREATED")
print("=" * 65)

print("Window array shape:", X_windows.shape)
print("Data type:", X_windows.dtype)
print("Total windows:", len(X_windows))
print("Window duration:", WINDOW_SAMPLES / FS, "seconds")
print("Samples per window:", WINDOW_SAMPLES)

print("\nClass distribution:")
print(window_metadata_df["fault_class"].value_counts().sort_index())

print("\nWindows by operating load:")
print(
    pd.crosstab(
        window_metadata_df["fault_class"],
        window_metadata_df["motor_load_hp"]
    )
)

print("\nRaw segmentation completed.")


In [ ]:

# ============================================================
# STEP 7C: SAVE RAW WINDOW DATASET
# ============================================================

from pathlib import Path
import joblib

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = (
    PROCESSED_DIR / "cwru_48k_raw_windows.joblib"
)

# Package data and its configuration together.
cwru_window_data = {
    "X_windows": X_windows,
    "metadata": window_metadata_df,
    "sampling_rate": FS,
    "window_samples": WINDOW_SAMPLES,
    "window_seconds": WINDOW_SAMPLES / FS,
    "label_mapping": LABEL_MAPPING
}

joblib.dump(
    cwru_window_data,
    OUTPUT_PATH,
    compress=3
)

# Verify that the saved artifact can be loaded.
loaded_data = joblib.load(OUTPUT_PATH)

assert loaded_data["X_windows"].shape == (140, 48000)
assert len(loaded_data["metadata"]) == 140

print("Dataset saved successfully.")
print("Location:", OUTPUT_PATH)
print("Shape:", loaded_data["X_windows"].shape)
print("Metadata rows:", len(loaded_data["metadata"]))
print("Sampling rate:", loaded_data["sampling_rate"])


## Step 8: Feature Engineering

### Step 8A: Time-Domain Features

Extract statistical features from the one-second vibration windows.

Each window contains 48,000 samples.

Features:
Mean, standard deviation, RMS, minimum, maximum,
peak-to-peak, skewness, kurtosis and crest factor.

No filtering or normalisation is performed.

In [ ]:

# ============================================================
# STEP 8A: TIME-DOMAIN FEATURE EXTRACTION — PILOT
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from scipy.stats import skew, kurtosis

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cwru_48k_raw_windows.joblib"
)

# Load saved window dataset.
saved_data = joblib.load(DATA_PATH)

X_windows = saved_data["X_windows"]
metadata = saved_data["metadata"].copy()

# Select the first 0 HP window from each class.
pilot_indices = (
    metadata[
        metadata["motor_load_hp"] == 0
    ]
    .groupby("fault_class", sort=True)
    .head(1)
    .index
    .tolist()
)

if len(pilot_indices) != 4:
    raise ValueError("Expected four pilot windows.")

# ------------------------------------------------------------
# Feature extraction function
# ------------------------------------------------------------

def extract_time_features(signal):

    signal = np.asarray(
        signal,
        dtype=np.float64
    )

    rms = np.sqrt(np.mean(signal ** 2))

    features = {
        "mean": np.mean(signal),
        "std": np.std(signal),
        "rms": rms,
        "min": np.min(signal),
        "max": np.max(signal),
        "peak_to_peak": np.ptp(signal),
        "skewness": skew(signal, bias=False),
        "kurtosis": kurtosis(
            signal,
            fisher=True,
            bias=False
        ),
        "crest_factor": (
            np.max(np.abs(signal)) / rms
            if rms > 0 else 0.0
        )
    }

    return features

# ------------------------------------------------------------
# Test on four representative windows
# ------------------------------------------------------------

pilot_rows = []

for idx in pilot_indices:

    signal = X_windows[idx]

    features = extract_time_features(signal)

    row = metadata.iloc[idx].to_dict()
    row.update(features)

    pilot_rows.append(row)

pilot_features_df = pd.DataFrame(pilot_rows)

display_columns = [
    "filename",
    "fault_class",
    "motor_load_hp",
    "mean",
    "std",
    "rms",
    "peak_to_peak",
    "skewness",
    "kurtosis",
    "crest_factor"
]

print("PILOT TIME-DOMAIN FEATURES")
print("=" * 90)

print(
    pilot_features_df[display_columns].to_string(
        index=False,
        formatters={
            col: "{:.5f}".format
            for col in display_columns
            if col not in [
                "filename",
                "fault_class",
                "motor_load_hp"
            ]
        }
    )
)

print("\nPilot windows processed:", len(pilot_features_df))
print("Features extracted:", len(extract_time_features(X_windows[0])))


In [ ]:

# ============================================================
# STEP 8A.2: TIME-DOMAIN FEATURES FOR ALL 140 WINDOWS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from scipy.stats import skew, kurtosis

# ------------------------------------------------------------
# 1. Load saved windows
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cwru_48k_raw_windows.joblib"
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "data"
    / "processed"
)

data = joblib.load(DATA_PATH)

X_windows = data["X_windows"]
metadata = data["metadata"].reset_index(drop=True).copy()

# ------------------------------------------------------------
# 2. Define time-domain feature extractor
# ------------------------------------------------------------

def extract_time_features(signal):

    signal = np.asarray(signal, dtype=np.float64)

    rms = np.sqrt(np.mean(signal ** 2))

    return {
        "mean": np.mean(signal),
        "std": np.std(signal),
        "rms": rms,
        "min": np.min(signal),
        "max": np.max(signal),
        "peak_to_peak": np.ptp(signal),
        "skewness": skew(signal, bias=False),
        "kurtosis": kurtosis(
            signal,
            fisher=True,
            bias=False
        ),
        "crest_factor": (
            np.max(np.abs(signal)) / rms
            if rms > 0 else 0.0
        )
    }

# ------------------------------------------------------------
# 3. Extract features from every window
# ------------------------------------------------------------

feature_rows = []

for index, signal in enumerate(X_windows):

    features = extract_time_features(signal)
    feature_rows.append(features)

time_features_df = pd.DataFrame(feature_rows)

# Combine original metadata and extracted features.
time_features_df = pd.concat(
    [metadata, time_features_df],
    axis=1
)

# ------------------------------------------------------------
# 4. Validate extracted features
# ------------------------------------------------------------

FEATURE_COLUMNS = [
    "mean",
    "std",
    "rms",
    "min",
    "max",
    "peak_to_peak",
    "skewness",
    "kurtosis",
    "crest_factor"
]

assert len(time_features_df) == 140
assert len(FEATURE_COLUMNS) == 9

feature_values = time_features_df[FEATURE_COLUMNS].to_numpy()

if not np.isfinite(feature_values).all():
    raise ValueError(
        "Non-finite values found in extracted features."
    )

print("Total windows:", len(time_features_df))
print("Time-domain features:", len(FEATURE_COLUMNS))
print("Feature matrix:", feature_values.shape)
print("All feature values finite:", np.isfinite(feature_values).all())

# ------------------------------------------------------------
# 5. Display feature summary by class
# ------------------------------------------------------------

print("\nTIME-DOMAIN FEATURE SUMMARY BY CLASS")

class_summary = (
    time_features_df
    .groupby("fault_class")[FEATURE_COLUMNS]
    .mean()
    .round(5)
)

print(class_summary.to_string())

print("\nWINDOW COUNTS BY CLASS")

print(
    time_features_df["fault_class"]
    .value_counts()
    .sort_index()
)

# ------------------------------------------------------------
# 6. Save results
# ------------------------------------------------------------

CSV_PATH = OUTPUT_DIR / "cwru_48k_time_features.csv"

JOBLIB_PATH = (
    OUTPUT_DIR / "cwru_48k_time_features.joblib"
)

time_features_df.to_csv(
    CSV_PATH,
    index=False
)

joblib.dump(
    time_features_df,
    JOBLIB_PATH,
    compress=3
)

print("\nTime-domain feature extraction completed.")
print("CSV:", CSV_PATH)
print("JOBLIB:", JOBLIB_PATH)


### Step 8B: Frequency-Domain Features

Extract frequency-domain features from the raw one-second
vibration windows using Welch's Power Spectral Density.

Frequency range: 0–24 kHz
Sampling frequency: 48 kHz

In [ ]:

# ============================================================
# STEP 8B.1: FREQUENCY-DOMAIN FEATURE EXTRACTION — PILOT
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from scipy.signal import welch

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cwru_48k_raw_windows.joblib"
)

data = joblib.load(DATA_PATH)

X_windows = data["X_windows"]
metadata = data["metadata"].reset_index(drop=True)

FS = 48000

# ------------------------------------------------------------
# Define frequency bands
# ------------------------------------------------------------

FREQUENCY_BANDS = {
    "0_500": (0, 500),
    "500_1000": (500, 1000),
    "1000_2000": (1000, 2000),
    "2000_4000": (2000, 4000),
    "4000_6000": (4000, 6000),
    "6000_10000": (6000, 10000),
    "10000_16000": (10000, 16000),
    "16000_24000": (16000, 24000)
}

# ------------------------------------------------------------
# Frequency feature extraction function
# ------------------------------------------------------------

def extract_frequency_features(signal):

    signal = np.asarray(
        signal,
        dtype=np.float64
    )

    frequencies, psd = welch(
        signal,
        fs=FS,
        window="hann",
        nperseg=8192,
        noverlap=4096,
        detrend="constant",
        scaling="density"
    )

    df = frequencies[1] - frequencies[0]

    # Ignore DC while calculating spectral descriptors.
    valid = frequencies >= 10

    f = frequencies[valid]
    p = psd[valid]

    total_power = np.sum(p) * df

    if total_power <= 0:
        raise ValueError("Non-positive spectral power.")

    # Dominant frequency
    dominant_frequency = f[np.argmax(p)]

    # Spectral centroid
    centroid = np.sum(f * p) / np.sum(p)

    # Spectral bandwidth
    bandwidth = np.sqrt(
        np.sum(((f - centroid) ** 2) * p)
        / np.sum(p)
    )

    # Spectral entropy
    probabilities = p / np.sum(p)
    probabilities = probabilities[probabilities > 0]

    entropy = -np.sum(
        probabilities * np.log2(probabilities)
    ) / np.log2(len(probabilities))

    # 95% spectral rolloff
    cumulative_power = np.cumsum(probabilities)

    rolloff_index = np.searchsorted(
        cumulative_power, 0.95
    )

    rolloff = f[min(rolloff_index, len(f) - 1)]

    features = {
        "dominant_frequency_hz": dominant_frequency,
        "spectral_centroid_hz": centroid,
        "spectral_bandwidth_hz": bandwidth,
        "spectral_entropy": entropy,
        "spectral_rolloff_95_hz": rolloff
    }

    # Log-transformed absolute band powers.
    for band_name, (low, high) in FREQUENCY_BANDS.items():

        mask = (
            (frequencies >= low) &
            (frequencies < high)
        )

        if high == FS / 2:
            mask |= frequencies == high

        band_power = np.sum(psd[mask]) * df

        features[f"log_bandpower_{band_name}"] = (
            np.log10(band_power + 1e-12)
        )

    return features

# ------------------------------------------------------------
# Test on one window per class at 0 HP
# ------------------------------------------------------------

pilot_indices = (
    metadata[metadata["motor_load_hp"] == 0]
    .groupby("fault_class", sort=True)
    .head(1)
    .index
    .tolist()
)

if len(pilot_indices) != 4:
    raise ValueError("Expected four pilot windows.")

pilot_rows = []

for idx in pilot_indices:

    features = extract_frequency_features(
        X_windows[idx]
    )

    row = metadata.iloc[idx].to_dict()
    row.update(features)

    pilot_rows.append(row)

pilot_frequency_df = pd.DataFrame(pilot_rows)

# ------------------------------------------------------------
# Display results
# ------------------------------------------------------------

FEATURE_COLUMNS = [
    "dominant_frequency_hz",
    "spectral_centroid_hz",
    "spectral_bandwidth_hz",
    "spectral_entropy",
    "spectral_rolloff_95_hz"
] + [
    f"log_bandpower_{name}"
    for name in FREQUENCY_BANDS
]

assert len(FEATURE_COLUMNS) == 13

assert np.isfinite(
    pilot_frequency_df[FEATURE_COLUMNS].to_numpy()
).all()

print("PILOT FREQUENCY-DOMAIN FEATURES")
print("=" * 100)

print(
    pilot_frequency_df[
        ["filename", "fault_class"] + FEATURE_COLUMNS
    ].to_string(index=False)
)

print("\nPilot windows:", len(pilot_frequency_df))
print("Frequency features:", len(FEATURE_COLUMNS))
print("All values finite: True")


### Step 8B.2: Frequency-Domain Features for All Windows

Apply the validated frequency-domain feature extraction method
to all 140 one-second vibration windows.

Extract 13 frequency-domain features per window.

In [ ]:

# ============================================================
# STEP 8B.2: FREQUENCY FEATURES FOR ALL 140 WINDOWS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
from scipy.signal import welch

# ------------------------------------------------------------
# 1. Load raw windows
# ------------------------------------------------------------

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT / "data" / "processed"
    / "cwru_48k_raw_windows.joblib"
)

data = joblib.load(DATA_PATH)

X_windows = data["X_windows"]
metadata = data["metadata"].reset_index(drop=True).copy()

FS = 48000

# ------------------------------------------------------------
# 2. Frequency bands
# ------------------------------------------------------------

FREQUENCY_BANDS = {
    "0_500": (0, 500),
    "500_1000": (500, 1000),
    "1000_2000": (1000, 2000),
    "2000_4000": (2000, 4000),
    "4000_6000": (4000, 6000),
    "6000_10000": (6000, 10000),
    "10000_16000": (10000, 16000),
    "16000_24000": (16000, 24000)
}

# ------------------------------------------------------------
# 3. Feature extraction function
# ------------------------------------------------------------

def extract_frequency_features(signal):

    frequencies, psd = welch(
        np.asarray(signal, dtype=np.float64),
        fs=FS,
        window="hann",
        nperseg=8192,
        noverlap=4096,
        detrend="constant",
        scaling="density"
    )

    df = frequencies[1] - frequencies[0]

    valid = frequencies >= 10

    f = frequencies[valid]
    p = psd[valid]

    power_sum = np.sum(p)

    if power_sum <= 0:
        raise ValueError("Non-positive spectral power.")

    # Spectral descriptors
    dominant_frequency = f[np.argmax(p)]

    centroid = np.sum(f * p) / power_sum

    bandwidth = np.sqrt(
        np.sum(((f - centroid) ** 2) * p)
        / power_sum
    )

    probabilities = p / power_sum

    positive_probabilities = probabilities[
        probabilities > 0
    ]

    entropy = (
        -np.sum(
            positive_probabilities
            * np.log2(positive_probabilities)
        )
        / np.log2(len(positive_probabilities))
    )

    cumulative_power = np.cumsum(probabilities)

    rolloff_index = np.searchsorted(
        cumulative_power, 0.95
    )

    rolloff = f[
        min(rolloff_index, len(f) - 1)
    ]

    features = {
        "dominant_frequency_hz": dominant_frequency,
        "spectral_centroid_hz": centroid,
        "spectral_bandwidth_hz": bandwidth,
        "spectral_entropy": entropy,
        "spectral_rolloff_95_hz": rolloff
    }

    # Eight logarithmic band-power features
    for name, (low, high) in FREQUENCY_BANDS.items():

        mask = (
            (frequencies >= low)
            & (frequencies < high)
        )

        if high == FS / 2:
            mask |= frequencies == FS / 2

        band_power = np.sum(psd[mask]) * df

        features[f"log_bandpower_{name}"] = (
            np.log10(band_power + 1e-12)
        )

    return features

# ------------------------------------------------------------
# 4. Extract features from all windows
# ------------------------------------------------------------

frequency_rows = []

for index, signal in enumerate(X_windows):

    features = extract_frequency_features(signal)

    frequency_rows.append(features)

    if (index + 1) % 20 == 0:
        print(
            f"Processed {index + 1}/{len(X_windows)} windows"
        )

frequency_features_df = pd.DataFrame(frequency_rows)

# Attach recording and class metadata.
frequency_features_df = pd.concat(
    [metadata, frequency_features_df],
    axis=1
)

# ------------------------------------------------------------
# 5. Validate features
# ------------------------------------------------------------

FEATURE_COLUMNS = [
    "dominant_frequency_hz",
    "spectral_centroid_hz",
    "spectral_bandwidth_hz",
    "spectral_entropy",
    "spectral_rolloff_95_hz"
] + [
    f"log_bandpower_{name}"
    for name in FREQUENCY_BANDS
]

values = frequency_features_df[FEATURE_COLUMNS].to_numpy()

assert frequency_features_df.shape[0] == 140
assert len(FEATURE_COLUMNS) == 13

if not np.isfinite(values).all():
    raise ValueError(
        "Non-finite values found in frequency features."
    )

# ------------------------------------------------------------
# 6. Summary by fault class
# ------------------------------------------------------------

class_summary = (
    frequency_features_df
    .groupby("fault_class")[FEATURE_COLUMNS]
    .mean()
    .round(5)
)

print("\nFREQUENCY FEATURE MATRIX:",
      values.shape)

print("\nCLASS-WISE FREQUENCY FEATURE SUMMARY:")
print(class_summary.to_string())

# ------------------------------------------------------------
# 7. Save results
# ------------------------------------------------------------

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CSV_PATH = (
    PROCESSED_DIR / "cwru_48k_frequency_features.csv"
)

JOBLIB_PATH = (
    PROCESSED_DIR / "cwru_48k_frequency_features.joblib"
)

frequency_features_df.to_csv(
    CSV_PATH,
    index=False
)

joblib.dump(
    frequency_features_df,
    JOBLIB_PATH,
    compress=3
)

print("\nFrequency feature extraction completed.")
print("Windows:", len(frequency_features_df))
print("Frequency features:", len(FEATURE_COLUMNS))
print("All values finite:", np.isfinite(values).all())
print("CSV:", CSV_PATH)
print("JOBLIB:", JOBLIB_PATH)


### Step 8C: Envelope-Spectrum Features

Extract bearing-specific envelope-spectrum energy features.

Processing:
1. Butterworth bandpass filtering.
2. Hilbert envelope extraction.
3. Welch envelope PSD estimation.
4. Energy measurement around fault-frequency harmonics.

Bearing: SKF 6205-2RS JEM
Sampling frequency: 48 kHz
Filter band: 500–5000 Hz
Envelope analysis range: 0–1000 Hz

In [ ]:

# ============================================================
# STEP 8C.1: ENVELOPE FEATURE CONFIGURATION
# ============================================================

import numpy as np

from scipy.signal import butter, sosfiltfilt, hilbert, welch

FS = 48000

LOWCUT = 500
HIGHCUT = 5000
FILTER_ORDER = 4

N_PER_SEG_ENV = 24000
N_OVERLAP_ENV = 12000

MAX_ENV_FREQ = 1000
PEAK_TOLERANCE_HZ = 5

HARMONICS = range(1, 6)

# CWRU SKF 6205-2RS JEM multipliers.
FAULT_MULTIPLIERS = {
    "bpfo": 3.5848,
    "bpfi": 5.4152,
    "bsf": 2.35675
}

# Design Butterworth bandpass filter.
sos = butter(
    FILTER_ORDER,
    [LOWCUT, HIGHCUT],
    btype="bandpass",
    fs=FS,
    output="sos"
)


def extract_envelope_features(signal, rpm):

    signal = np.asarray(
        signal,
        dtype=np.float64
    )

    if not np.isfinite(signal).all():
        raise ValueError("Signal contains non-finite values.")

    # 1. Bandpass filtering
    filtered = sosfiltfilt(sos, signal)

    # 2. Hilbert envelope
    envelope = np.abs(hilbert(filtered))

    envelope = envelope - np.mean(envelope)

    # 3. Envelope PSD using Welch's method
    frequencies, psd = welch(
        envelope,
        fs=FS,
        window="hann",
        nperseg=N_PER_SEG_ENV,
        noverlap=N_OVERLAP_ENV,
        detrend="constant",
        scaling="density"
    )

    # Restrict analysis to 1–1000 Hz.
    mask = (
        (frequencies >= 1)
        & (frequencies <= MAX_ENV_FREQ)
    )

    freq = frequencies[mask]
    power = psd[mask]

    df = frequencies[1] - frequencies[0]

    shaft_frequency = rpm / 60.0

    features = {}

    # 4. Calculate energy near each fault harmonic.
    for fault_name, multiplier in FAULT_MULTIPLIERS.items():

        fundamental = multiplier * shaft_frequency

        for harmonic in HARMONICS:

            expected = fundamental * harmonic

            local_mask = (
                (freq >= expected - PEAK_TOLERANCE_HZ)
                & (freq <= expected + PEAK_TOLERANCE_HZ)
            )

            if np.any(local_mask):

                local_energy = (
                    np.sum(power[local_mask]) * df
                )

            else:
                local_energy = 0.0

            feature_name = (
                f"log_env_{fault_name}_h{harmonic}_power"
            )

            features[feature_name] = np.log10(
                local_energy + 1e-12
            )

    return features, freq, power


In [ ]:

# ============================================================
# STEP 8C.2: ENVELOPE FEATURE PILOT TEST
# ============================================================

from pathlib import Path
import pandas as pd
import joblib
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cwru_48k_raw_windows.joblib"
)

data = joblib.load(DATA_PATH)

X_windows = data["X_windows"]
metadata = data["metadata"].reset_index(drop=True)

# Select one 0 HP window per class.
pilot_indices = (
    metadata[metadata["motor_load_hp"] == 0]
    .groupby("fault_class", sort=True)
    .head(1)
    .index.tolist()
)

pilot_rows = []

fig, axes = plt.subplots(
    2, 2,
    figsize=(14, 8),
    sharex=True
)

axes = axes.flatten()

for ax, idx in zip(axes, pilot_indices):

    row = metadata.iloc[idx]

    fault_class = row["fault_class"]
    rpm = float(row["rpm"])

    signal = X_windows[idx]

    features, freq, power = extract_envelope_features(
        signal, rpm
    )

    result = row.to_dict()
    result.update(features)

    pilot_rows.append(result)

    ax.plot(
        freq,
        10 * np.log10(np.maximum(power, 1e-18))
    )

    # Mark relevant theoretical fault harmonics.
    fault_key = {
        "Inner": "bpfi",
        "Outer": "bpfo",
        "Ball": "bsf"
    }.get(fault_class)

    if fault_key is not None:

        fundamental = (
            FAULT_MULTIPLIERS[fault_key]
            * rpm / 60.0
        )

        for harmonic in HARMONICS:

            ax.axvline(
                fundamental * harmonic,
                linestyle="--",
                alpha=0.5
            )

    ax.set_title(f"{fault_class} — {rpm:.0f} RPM")
    ax.set_xlabel("Frequency (Hz)")
    ax.set_ylabel("Envelope PSD (dB/Hz)")
    ax.set_xlim(0, MAX_ENV_FREQ)
    ax.grid(True)

fig.suptitle(
    "CWRU 48 kHz Pilot Envelope Spectra",
    fontsize=14
)

plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# Display extracted features
# ------------------------------------------------------------

pilot_envelope_df = pd.DataFrame(pilot_rows)

ENVELOPE_COLUMNS = [
    f"log_env_{fault}_h{h}_power"
    for fault in FAULT_MULTIPLIERS
    for h in HARMONICS
]

print("\nPILOT ENVELOPE FEATURES")
print("=" * 85)

print(
    pilot_envelope_df[
        ["filename", "fault_class", "rpm"]
        + ENVELOPE_COLUMNS
    ].to_string(index=False)
)

values = pilot_envelope_df[
    ENVELOPE_COLUMNS
].to_numpy()

print("\nPilot windows:", len(pilot_envelope_df))
print("Envelope features:", len(ENVELOPE_COLUMNS))
print("All values finite:", np.isfinite(values).all())


### Step 8C.3: Envelope Features for All Windows

Apply the validated envelope feature extraction method
to all 140 one-second vibration windows.

Extract:
- 5 BPFO harmonic energy features
- 5 BPFI harmonic energy features
- 5 BSF harmonic energy features

RPM is taken from the corresponding recording metadata.

In [ ]:

# ============================================================
# STEP 8C.3: ENVELOPE FEATURES FOR ALL 140 WINDOWS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cwru_48k_raw_windows.joblib"
)

# Load raw windows and metadata.
data = joblib.load(DATA_PATH)

X_windows = data["X_windows"]
metadata = data["metadata"].reset_index(drop=True).copy()

assert len(X_windows) == len(metadata) == 140

# ------------------------------------------------------------
# Extract envelope features
# ------------------------------------------------------------

envelope_rows = []

for index, signal in enumerate(X_windows):

    rpm = float(metadata.loc[index, "rpm"])

    features, _, _ = extract_envelope_features(
        signal,
        rpm
    )

    envelope_rows.append(features)

    if (index + 1) % 20 == 0:
        print(
            f"Processed {index + 1}/{len(X_windows)} windows"
        )

# ------------------------------------------------------------
# Create feature DataFrame
# ------------------------------------------------------------

envelope_features_df = pd.DataFrame(envelope_rows)

envelope_features_df = pd.concat(
    [metadata, envelope_features_df],
    axis=1
)

ENVELOPE_COLUMNS = [
    f"log_env_{fault}_h{harmonic}_power"
    for fault in ["bpfo", "bpfi", "bsf"]
    for harmonic in range(1, 6)
]

# ------------------------------------------------------------
# Validate results
# ------------------------------------------------------------

feature_values = (
    envelope_features_df[ENVELOPE_COLUMNS]
    .to_numpy()
)

assert feature_values.shape == (140, 15)

if not np.isfinite(feature_values).all():
    raise ValueError(
        "Non-finite values found in envelope features."
    )

print("\nEnvelope feature matrix:", feature_values.shape)
print("All values finite:", np.isfinite(feature_values).all())

# ------------------------------------------------------------
# Class-wise summary
# ------------------------------------------------------------

class_summary = (
    envelope_features_df
    .groupby("fault_class")[ENVELOPE_COLUMNS]
    .mean()
    .round(4)
)

print("\nENVELOPE FEATURE SUMMARY BY CLASS")
print(class_summary.to_string())

# ------------------------------------------------------------
# Save results
# ------------------------------------------------------------

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

CSV_PATH = (
    PROCESSED_DIR / "cwru_48k_envelope_features.csv"
)

JOBLIB_PATH = (
    PROCESSED_DIR / "cwru_48k_envelope_features.joblib"
)

envelope_features_df.to_csv(
    CSV_PATH,
    index=False
)

joblib.dump(
    envelope_features_df,
    JOBLIB_PATH,
    compress=3
)

print("\nEnvelope feature extraction completed.")
print("Windows:", len(envelope_features_df))
print("Features:", len(ENVELOPE_COLUMNS))
print("CSV:", CSV_PATH)
print("JOBLIB:", JOBLIB_PATH)


### Step 8D: Integration of All Extracted Features

Combine time-domain, frequency-domain and envelope-spectrum
features into one dataset.

Verify:
- Consistent recording and window identifiers.
- Consistent metadata across feature tables.
- Complete 37-feature matrix.
- Finite numerical values.

Save the integrated feature dataset.

In [ ]:

# ============================================================
# STEP 8D: LOAD AND COMBINE FEATURE DATASETS
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

# Load the three existing feature tables.
time_df = joblib.load(
    PROCESSED_DIR / "cwru_48k_time_features.joblib"
)

frequency_df = joblib.load(
    PROCESSED_DIR / "cwru_48k_frequency_features.joblib"
)

envelope_df = joblib.load(
    PROCESSED_DIR / "cwru_48k_envelope_features.joblib"
)

print("Loaded feature tables:")
print("Time:", time_df.shape)
print("Frequency:", frequency_df.shape)
print("Envelope:", envelope_df.shape)


In [ ]:

# ============================================================
# STEP 8D.2: VERIFY AND COMBINE FEATURES
# ============================================================

KEY_COLUMNS = ["filename", "window_index"]

METADATA_COLUMNS = [
    "filename",
    "fault_class",
    "label",
    "motor_load_hp",
    "rpm",
    "rpm_source",
    "window_index",
    "start_sample",
    "end_sample",
    "start_time_seconds",
    "end_time_seconds"
]

TIME_FEATURES = [
    "mean", "std", "rms", "min", "max",
    "peak_to_peak", "skewness", "kurtosis",
    "crest_factor"
]

FREQUENCY_FEATURES = [
    "dominant_frequency_hz",
    "spectral_centroid_hz",
    "spectral_bandwidth_hz",
    "spectral_entropy",
    "spectral_rolloff_95_hz"
] + [
    f"log_bandpower_{name}"
    for name in [
        "0_500", "500_1000", "1000_2000",
        "2000_4000", "4000_6000",
        "6000_10000", "10000_16000",
        "16000_24000"
    ]
]

ENVELOPE_FEATURES = [
    f"log_env_{fault}_h{h}_power"
    for fault in ["bpfo", "bpfi", "bsf"]
    for h in range(1, 6)
]

ALL_FEATURES = (
    TIME_FEATURES
    + FREQUENCY_FEATURES
    + ENVELOPE_FEATURES
)

# Verify expected columns exist.
for name, df in [
    ("Time", time_df),
    ("Frequency", frequency_df),
    ("Envelope", envelope_df)
]:

    missing = [
        col for col in METADATA_COLUMNS
        if col not in df.columns
    ]

    if missing:
        raise ValueError(
            f"{name} table missing metadata: {missing}"
        )

# Sort each table consistently.
tables = [
    time_df,
    frequency_df,
    envelope_df
]

for df in tables:

    df.sort_values(
        KEY_COLUMNS,
        inplace=True
    )

    df.reset_index(drop=True, inplace=True)

    if len(df) != 140:
        raise ValueError(
            "Unexpected number of windows."
        )

    if df.duplicated(KEY_COLUMNS).any():
        raise ValueError(
            "Duplicate recording/window identifiers found."
        )

# Verify metadata alignment.
reference_metadata = (
    time_df[METADATA_COLUMNS]
    .reset_index(drop=True)
)

for name, df in [
    ("Frequency", frequency_df),
    ("Envelope", envelope_df)
]:

    try:
        pd.testing.assert_frame_equal(
            reference_metadata,
            df[METADATA_COLUMNS].reset_index(drop=True),
            check_dtype=False
        )

        print(f"{name} metadata: MATCH")

    except AssertionError:
        raise ValueError(
            f"{name} metadata does not match."
        )

# Build combined feature table.
combined_df = time_df[
    METADATA_COLUMNS + TIME_FEATURES
].copy()

combined_df = combined_df.merge(
    frequency_df[
        KEY_COLUMNS + FREQUENCY_FEATURES
    ],
    on=KEY_COLUMNS,
    how="inner",
    validate="one_to_one"
)

combined_df = combined_df.merge(
    envelope_df[
        KEY_COLUMNS + ENVELOPE_FEATURES
    ],
    on=KEY_COLUMNS,
    how="inner",
    validate="one_to_one"
)

# Verify combined matrix.
X_features = combined_df[ALL_FEATURES].to_numpy(
    dtype=np.float64
)

assert combined_df.shape[0] == 140
assert len(ALL_FEATURES) == 37
assert X_features.shape == (140, 37)

if not np.isfinite(X_features).all():
    raise ValueError(
        "Non-finite values in combined feature matrix."
    )

print("\nCOMBINED FEATURE DATASET")
print("=" * 60)

print("Windows:", len(combined_df))
print("Time features:", len(TIME_FEATURES))
print("Frequency features:", len(FREQUENCY_FEATURES))
print("Envelope features:", len(ENVELOPE_FEATURES))
print("Total features:", len(ALL_FEATURES))
print("Feature matrix shape:", X_features.shape)
print("All values finite:", np.isfinite(X_features).all())

print("\nClass distribution:")
print(combined_df["fault_class"].value_counts().sort_index())


In [ ]:

# ============================================================
# STEP 8D.3: SAVE COMBINED FEATURE DATASET
# ============================================================

CSV_PATH = (
    PROCESSED_DIR / "cwru_48k_combined_features.csv"
)

JOBLIB_PATH = (
    PROCESSED_DIR / "cwru_48k_combined_features.joblib"
)

combined_df.to_csv(
    CSV_PATH,
    index=False
)

combined_data = {
    "features": combined_df[ALL_FEATURES].to_numpy(
        dtype=np.float32
    ),
    "metadata": combined_df[METADATA_COLUMNS].copy(),
    "feature_names": ALL_FEATURES,
    "sampling_rate": 48000,
    "window_samples": 48000,
    "window_seconds": 1.0
}

joblib.dump(
    combined_data,
    JOBLIB_PATH,
    compress=3
)

print("Combined dataset saved successfully.")
print("CSV:", CSV_PATH)
print("JOBLIB:", JOBLIB_PATH)
print("Feature shape:", combined_data["features"].shape)


## Step 9: Classification Experiment

### Step 9A: Dataset Loading and Evaluation Preparation

Load the integrated 37-feature CWRU dataset.

Verify:
- Feature dimensions and data quality.
- Class distribution.
- Operating-load distribution.
- Recording-level separation for Leave-One-Load-Out validation.

No scaling, feature selection or model training is performed.

In [ ]:

# ============================================================
# STEP 9A: LOAD AND VERIFY CLASSIFICATION DATASET
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cwru_48k_combined_features.joblib"
)

data = joblib.load(DATA_PATH)

X = np.asarray(data["features"], dtype=np.float64)
meta = data["metadata"].reset_index(drop=True).copy()

FEATURE_NAMES = data["feature_names"]

# Verify dataset.
assert X.shape == (140, 37)
assert len(meta) == 140
assert len(FEATURE_NAMES) == 37
assert np.isfinite(X).all()

assert meta["filename"].nunique() == 16
assert meta["window_index"].notna().all()

# Ensure each recording-window pair is unique.
assert not meta.duplicated(
    ["filename", "window_index"]
).any()

# Check class and load distributions.
print("Feature matrix:", X.shape)
print("Number of features:", len(FEATURE_NAMES))
print("All values finite:", np.isfinite(X).all())

print("\nClass distribution:")
print(meta["fault_class"].value_counts().sort_index())

print("\nOperating-load distribution:")
print(meta["motor_load_hp"].value_counts().sort_index())

print("\nClass versus operating load:")
print(
    pd.crosstab(
        meta["fault_class"],
        meta["motor_load_hp"]
    )
)

print("\nDataset verification completed.")


In [ ]:

# ============================================================
# STEP 9B: LEAVE-ONE-LOAD-OUT FOLD VERIFICATION
# ============================================================

fold_rows = []

for held_out_load in sorted(meta["motor_load_hp"].unique()):

    test_mask = (
        meta["motor_load_hp"] == held_out_load
    )

    train_mask = ~test_mask

    train_meta = meta.loc[train_mask]
    test_meta = meta.loc[test_mask]

    # Verify complete recording separation.
    train_files = set(train_meta["filename"])
    test_files = set(test_meta["filename"])

    assert train_files.isdisjoint(test_files)

    # Verify all four classes in both partitions.
    assert train_meta["fault_class"].nunique() == 4
    assert test_meta["fault_class"].nunique() == 4

    # Each class must be equally represented within a partition.
    assert train_meta["fault_class"].value_counts().nunique() == 1
    assert test_meta["fault_class"].value_counts().nunique() == 1

    fold_rows.append({
        "held_out_load_hp": held_out_load,
        "train_windows": len(train_meta),
        "test_windows": len(test_meta),
        "train_recordings": train_meta["filename"].nunique(),
        "test_recordings": test_meta["filename"].nunique(),
        "train_classes": train_meta["fault_class"].nunique(),
        "test_classes": test_meta["fault_class"].nunique(),
        "recording_overlap": len(
            train_files.intersection(test_files)
        )
    })

    print(
        f"Fold {len(fold_rows)} | "
        f"Test load: {held_out_load} HP | "
        f"Train: {len(train_meta)} windows | "
        f"Test: {len(test_meta)} windows | "
        f"Recording overlap: {len(train_files & test_files)}"
    )

fold_summary = pd.DataFrame(fold_rows)

print("\nFOLD SUMMARY")
print(fold_summary.to_string(index=False))

assert len(fold_summary) == 4
assert (fold_summary["recording_overlap"] == 0).all()

print("\nAll four evaluation folds verified.")


In [ ]:

# ============================================================
# STEP 9C.1: LOAD DATA FOR LOGISTIC REGRESSION
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cwru_48k_combined_features.joblib"
)

data = joblib.load(DATA_PATH)

X = np.asarray(
    data["features"],
    dtype=np.float64
)

meta = data["metadata"].reset_index(drop=True).copy()

y = meta["label"].to_numpy(dtype=int)

# Establish label names from the saved metadata.
label_table = (
    meta[["label", "fault_class"]]
    .drop_duplicates()
    .sort_values("label")
)

LABEL_NAMES = dict(
    zip(
        label_table["label"].astype(int),
        label_table["fault_class"]
    )
)

CLASSES = sorted(LABEL_NAMES)

assert X.shape == (140, 37)
assert len(y) == 140
assert len(CLASSES) == 4
assert np.isfinite(X).all()

print("Feature shape:", X.shape)
print("Label shape:", y.shape)
print("Class mapping:", LABEL_NAMES)
print("Dataset loaded successfully.")


In [ ]:

# ============================================================
# STEP 9C.2: FOUR-FOLD LOGISTIC REGRESSION EVALUATION
# Leave-One-Operating-Load-Out
# ============================================================

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

fold_metrics = []
oof_predictions = []

for fold_number, held_out_load in enumerate(
    sorted(meta["motor_load_hp"].unique()),
    start=1
):

    # --------------------------------------------------------
    # 1. Create training and test partitions
    # --------------------------------------------------------

    test_mask = (
        meta["motor_load_hp"].to_numpy()
        == held_out_load
    )

    train_mask = ~test_mask

    X_train, X_test = X[train_mask], X[test_mask]
    y_train, y_test = y[train_mask], y[test_mask]

    train_meta = meta.loc[train_mask]
    test_meta = meta.loc[test_mask]

    # Verify recording-level separation.
    assert set(train_meta["filename"]).isdisjoint(
        set(test_meta["filename"])
    )

    # --------------------------------------------------------
    # 2. Create pipeline
    # --------------------------------------------------------

    model = Pipeline([
        ("scaler", StandardScaler()),

        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            solver="lbfgs",
            random_state=42
        ))
    ])

    # --------------------------------------------------------
    # 3. Train model
    # --------------------------------------------------------

    model.fit(X_train, y_train)

    # --------------------------------------------------------
    # 4. Predict
    # --------------------------------------------------------

    predictions = model.predict(X_test)

    probabilities = model.predict_proba(X_test)

    model_classes = (
        model.named_steps["classifier"].classes_
    )

    # Align probability columns with CLASSES.
    column_order = [
        list(model_classes).index(c)
        for c in CLASSES
    ]

    probabilities = probabilities[:, column_order]

    # --------------------------------------------------------
    # 5. Calculate evaluation metrics
    # --------------------------------------------------------

    metrics = {
        "fold": fold_number,
        "held_out_load_hp": int(held_out_load),
        "train_windows": len(X_train),
        "test_windows": len(X_test),

        "accuracy": accuracy_score(
            y_test, predictions
        ),

        "balanced_accuracy": balanced_accuracy_score(
            y_test, predictions
        ),

        "macro_precision": precision_score(
            y_test,
            predictions,
            labels=CLASSES,
            average="macro",
            zero_division=0
        ),

        "macro_recall": recall_score(
            y_test,
            predictions,
            labels=CLASSES,
            average="macro",
            zero_division=0
        ),

        "macro_f1": f1_score(
            y_test,
            predictions,
            labels=CLASSES,
            average="macro",
            zero_division=0
        ),

        "weighted_f1": f1_score(
            y_test,
            predictions,
            labels=CLASSES,
            average="weighted",
            zero_division=0
        ),

        "roc_auc_ovr_macro": roc_auc_score(
            y_test,
            probabilities,
            labels=CLASSES,
            multi_class="ovr",
            average="macro"
        )
    }

    fold_metrics.append(metrics)

    # --------------------------------------------------------
    # 6. Save out-of-fold predictions in memory
    # --------------------------------------------------------

    result = test_meta[
        [
            "filename",
            "fault_class",
            "label",
            "motor_load_hp",
            "window_index"
        ]
    ].copy().reset_index(drop=True)

    result["predicted_label"] = predictions
    result["predicted_class"] = [
        LABEL_NAMES[int(value)]
        for value in predictions
    ]

    result["fold"] = fold_number

    for index, class_id in enumerate(CLASSES):

        class_name = LABEL_NAMES[class_id]

        result[
            f"probability_{class_name}"
        ] = probabilities[:, index]

    oof_predictions.append(result)

    # --------------------------------------------------------
    # 7. Display fold results
    # --------------------------------------------------------

    print("\n" + "=" * 65)
    print(
        f"FOLD {fold_number} — TEST LOAD: "
        f"{held_out_load} HP"
    )
    print("=" * 65)

    print("Training windows:", len(X_train))
    print("Test windows:", len(X_test))

    for metric_name, value in metrics.items():

        if isinstance(value, (int, np.integer)):
            continue

        if metric_name in [
            "fold",
            "held_out_load_hp",
            "train_windows",
            "test_windows"
        ]:
            continue

        print(f"{metric_name}: {value:.4f}")

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_test,
            predictions,
            labels=CLASSES
        )
    )

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            predictions,
            labels=CLASSES,
            target_names=[
                LABEL_NAMES[c] for c in CLASSES
            ],
            zero_division=0
        )
    )

# ------------------------------------------------------------
# 8. Combine out-of-fold results
# ------------------------------------------------------------

fold_metrics_df = pd.DataFrame(fold_metrics)

oof_predictions_df = pd.concat(
    oof_predictions,
    ignore_index=True
)

assert len(oof_predictions_df) == 140
assert not oof_predictions_df.duplicated(
    ["filename", "window_index"]
).any()

print("\n" + "=" * 65)
print("FOUR-FOLD EVALUATION SUMMARY")
print("=" * 65)

print(
    fold_metrics_df.round(4).to_string(index=False)
)

print("\nOut-of-fold predictions:", len(oof_predictions_df))

print("\nLogistic Regression evaluation completed.")


In [ ]:

# ============================================================
# STEP 9D: OUT-OF-FOLD PREDICTION AUDIT
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

OUTPUT_DIR = (
    PROJECT_ROOT
    / "results"
    / "cwru_classification"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Check the previously generated predictions.
oof = oof_predictions_df.copy()

assert len(oof) == 140
assert oof["filename"].nunique() == 16
assert not oof.duplicated(
    ["filename", "window_index"]
).any()

oof["correct"] = (
    oof["label"] == oof["predicted_label"]
)

# Extract predicted probability of the true class.
oof["true_class_probability"] = [
    row[f"probability_{LABEL_NAMES[int(row['label'])]}"]
    for _, row in oof.iterrows()
]

# ------------------------------------------------------------
# Pooled metrics
# ------------------------------------------------------------

y_true = oof["label"].to_numpy()
y_pred = oof["predicted_label"].to_numpy()

print("=" * 65)
print("POOLED OUT-OF-FOLD RESULTS")
print("=" * 65)

print("Accuracy:", accuracy_score(y_true, y_pred))
print(
    "Balanced accuracy:",
    balanced_accuracy_score(y_true, y_pred)
)
print(
    "Macro F1:",
    f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )
)

print("\nPooled confusion matrix:")

print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=CLASSES
    )
)

print("\nPooled classification report:")

print(
    classification_report(
        y_true,
        y_pred,
        labels=CLASSES,
        target_names=[
            LABEL_NAMES[c] for c in CLASSES
        ],
        zero_division=0
    )
)

# ------------------------------------------------------------
# Per-recording audit
# ------------------------------------------------------------

recording_audit = (
    oof.groupby(
        ["filename", "fault_class", "motor_load_hp"]
    )
    .agg(
        windows=("correct", "size"),
        accuracy=("correct", "mean"),
        mean_true_class_probability=(
            "true_class_probability", "mean"
        ),
        minimum_true_class_probability=(
            "true_class_probability", "min"
        )
    )
    .reset_index()
    .sort_values(
        ["motor_load_hp", "fault_class"]
    )
)

print("\nPER-RECORDING PERFORMANCE")
print(
    recording_audit.to_string(
        index=False,
        formatters={
            "accuracy": "{:.3f}".format,
            "mean_true_class_probability": "{:.4f}".format,
            "minimum_true_class_probability": "{:.4f}".format
        }
    )
)

# ------------------------------------------------------------
# Save all evaluation results
# ------------------------------------------------------------

oof.to_csv(
    OUTPUT_DIR / "cwru_48k_oof_predictions.csv",
    index=False
)

fold_metrics_df.to_csv(
    OUTPUT_DIR / "cwru_48k_fold_metrics.csv",
    index=False
)

recording_audit.to_csv(
    OUTPUT_DIR / "cwru_48k_recording_audit.csv",
    index=False
)

joblib.dump(
    {
        "fold_metrics": fold_metrics_df,
        "oof_predictions": oof,
        "recording_audit": recording_audit
    },
    OUTPUT_DIR / "cwru_48k_evaluation_results.joblib",
    compress=3
)

assert oof["correct"].all()

print("\nAll 140 predictions verified.")
print("All 16 recording accuracies:")
print(recording_audit["accuracy"].value_counts().to_dict())

print("\nEvaluation results saved to:", OUTPUT_DIR)


### Step 9E: Feature-Group Ablation

Evaluate the contribution of time-domain, frequency-domain
and envelope-spectrum features.

Use identical Leave-One-Load-Out folds and Logistic Regression
configuration for every feature combination.

StandardScaler is fitted only on training data in each fold.

In [ ]:

# ============================================================
# STEP 9E.1: DEFINE FEATURE GROUPS
# ============================================================

from pathlib import Path
import numpy as np
import joblib

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cwru_48k_combined_features.joblib"
)

data = joblib.load(DATA_PATH)

X_all = np.asarray(data["features"], dtype=np.float64)
meta = data["metadata"].reset_index(drop=True).copy()
feature_names = list(data["feature_names"])

# ------------------------------------------------------------
# Feature groups
# ------------------------------------------------------------

TIME_FEATURES = [
    "mean", "std", "rms", "min", "max",
    "peak_to_peak", "skewness", "kurtosis",
    "crest_factor"
]

FREQUENCY_FEATURES = [
    "dominant_frequency_hz",
    "spectral_centroid_hz",
    "spectral_bandwidth_hz",
    "spectral_entropy",
    "spectral_rolloff_95_hz"
] + [
    f"log_bandpower_{name}"
    for name in [
        "0_500", "500_1000", "1000_2000",
        "2000_4000", "4000_6000",
        "6000_10000", "10000_16000",
        "16000_24000"
    ]
]

ENVELOPE_FEATURES = [
    f"log_env_{fault}_h{h}_power"
    for fault in ["bpfo", "bpfi", "bsf"]
    for h in range(1, 6)
]

# ------------------------------------------------------------
# Seven experimental configurations
# ------------------------------------------------------------

FEATURE_GROUPS = {
    "Time": TIME_FEATURES,

    "Frequency": FREQUENCY_FEATURES,

    "Envelope": ENVELOPE_FEATURES,

    "Time + Frequency":
        TIME_FEATURES + FREQUENCY_FEATURES,

    "Time + Envelope":
        TIME_FEATURES + ENVELOPE_FEATURES,

    "Frequency + Envelope":
        FREQUENCY_FEATURES + ENVELOPE_FEATURES,

    "All 37 Features":
        TIME_FEATURES
        + FREQUENCY_FEATURES
        + ENVELOPE_FEATURES
}

# ------------------------------------------------------------
# Verify configurations
# ------------------------------------------------------------

assert X_all.shape == (140, 37)
assert len(feature_names) == 37
assert len(meta) == 140

for name, features in FEATURE_GROUPS.items():

    missing = [
        feature for feature in features
        if feature not in feature_names
    ]

    if missing:
        raise ValueError(
            f"{name} has missing features: {missing}"
        )

    assert len(features) == len(set(features))

print("FEATURE-GROUP CONFIGURATIONS")
print("=" * 60)

for name, features in FEATURE_GROUPS.items():
    print(f"{name:<25} {len(features):>3} features")

print("\nAll seven configurations verified.")


In [ ]:

# ============================================================
# STEP 9E.2: FOUR-FOLD FEATURE-GROUP ABLATION
# ============================================================

import pandas as pd

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    roc_auc_score
)

LABELS = sorted(meta["label"].unique())

fold_ablation_rows = []
ablation_oof_rows = []

# Map each feature name to its array position.
feature_index = {
    name: index
    for index, name in enumerate(feature_names)
}

for group_name, selected_features in FEATURE_GROUPS.items():

    selected_indices = [
        feature_index[name]
        for name in selected_features
    ]

    X_group = X_all[:, selected_indices]

    print("\n" + "=" * 65)
    print("FEATURE CONFIGURATION:", group_name)
    print("Number of features:", len(selected_features))
    print("=" * 65)

    for fold_number, held_out_load in enumerate(
        sorted(meta["motor_load_hp"].unique()),
        start=1
    ):

        test_mask = (
            meta["motor_load_hp"].to_numpy()
            == held_out_load
        )

        train_mask = ~test_mask

        X_train = X_group[train_mask]
        X_test = X_group[test_mask]

        y_train = meta.loc[
            train_mask, "label"
        ].to_numpy(dtype=int)

        y_test = meta.loc[
            test_mask, "label"
        ].to_numpy(dtype=int)

        train_meta = meta.loc[train_mask]
        test_meta = meta.loc[test_mask]

        # Prevent recording overlap.
        assert set(
            train_meta["filename"]
        ).isdisjoint(
            set(test_meta["filename"])
        )

        # Same model configuration as baseline.
        model = Pipeline([
            ("scaler", StandardScaler()),

            ("classifier", LogisticRegression(
                class_weight="balanced",
                max_iter=3000,
                solver="lbfgs",
                random_state=42
            ))
        ])

        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        probabilities = model.predict_proba(X_test)

        # Align probability columns with global labels.
        fitted_classes = (
            model.named_steps["classifier"].classes_
        )

        order = [
            list(fitted_classes).index(label)
            for label in LABELS
        ]

        probabilities = probabilities[:, order]

        # Calculate metrics.
        metrics = {
            "feature_group": group_name,
            "feature_count": len(selected_features),
            "held_out_load_hp": int(held_out_load),

            "accuracy": accuracy_score(
                y_test, predictions
            ),

            "balanced_accuracy": balanced_accuracy_score(
                y_test, predictions
            ),

            "macro_f1": f1_score(
                y_test,
                predictions,
                labels=LABELS,
                average="macro",
                zero_division=0
            ),

            "roc_auc_ovr_macro": roc_auc_score(
                y_test,
                probabilities,
                labels=LABELS,
                multi_class="ovr",
                average="macro"
            )
        }

        fold_ablation_rows.append(metrics)

        # Keep out-of-fold predictions.
        result = test_meta[
            [
                "filename",
                "fault_class",
                "label",
                "motor_load_hp",
                "window_index"
            ]
        ].copy().reset_index(drop=True)

        result["predicted_label"] = predictions
        result["feature_group"] = group_name
        result["fold"] = fold_number

        ablation_oof_rows.append(result)

        print(
            f"Load {held_out_load} HP | "
            f"Accuracy: {metrics['accuracy']:.4f} | "
            f"Macro F1: {metrics['macro_f1']:.4f}"
        )

print("\nAblation model training completed.")

fold_ablation_df = pd.DataFrame(fold_ablation_rows)
ablation_predictions_df = pd.concat(
    ablation_oof_rows,
    ignore_index=True
)

assert len(fold_ablation_df) == 28
assert len(ablation_predictions_df) == 980

print("Models evaluated:", len(fold_ablation_df))
print("Total out-of-fold prediction records:",
      len(ablation_predictions_df))


In [ ]:

# ============================================================
# STEP 9E.3: ABLATION SUMMARY AND SAVE
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

OUTPUT_DIR = (
    PROJECT_ROOT / "results" / "cwru_classification"
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Mean and standard deviation across four loads
# ------------------------------------------------------------

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_f1",
    "roc_auc_ovr_macro"
]

fold_summary = (
    fold_ablation_df
    .groupby("feature_group")[metric_columns]
    .agg(["mean", "std"])
)

# ------------------------------------------------------------
# 2. Pooled out-of-fold metrics
# ------------------------------------------------------------

pooled_rows = []

for group_name, group in ablation_predictions_df.groupby(
    "feature_group",
    sort=False
):

    y_true = group["label"].to_numpy()
    y_pred = group["predicted_label"].to_numpy()

    pooled_rows.append({
        "feature_group": group_name,
        "pooled_accuracy": accuracy_score(y_true, y_pred),
        "pooled_balanced_accuracy": balanced_accuracy_score(
            y_true, y_pred
        ),
        "pooled_macro_f1": f1_score(
            y_true,
            y_pred,
            labels=LABELS,
            average="macro",
            zero_division=0
        )
    })

pooled_summary = pd.DataFrame(pooled_rows)

# ------------------------------------------------------------
# 3. Assemble readable summary
# ------------------------------------------------------------

summary_rows = []

for group_name, features in FEATURE_GROUPS.items():

    fold_data = fold_ablation_df[
        fold_ablation_df["feature_group"] == group_name
    ]

    pooled_data = pooled_summary[
        pooled_summary["feature_group"] == group_name
    ].iloc[0]

    row = {
        "feature_group": group_name,
        "feature_count": len(features)
    }

    for metric in metric_columns:
        row[f"mean_{metric}"] = fold_data[metric].mean()
        row[f"std_{metric}"] = fold_data[metric].std()

    row["pooled_accuracy"] = pooled_data["pooled_accuracy"]
    row["pooled_balanced_accuracy"] = (
        pooled_data["pooled_balanced_accuracy"]
    )
    row["pooled_macro_f1"] = pooled_data["pooled_macro_f1"]

    summary_rows.append(row)

ablation_summary_df = pd.DataFrame(summary_rows)

print("=" * 100)
print("CWRU FEATURE-GROUP ABLATION RESULTS")
print("=" * 100)

print(
    ablation_summary_df.round(4).to_string(index=False)
)

# ------------------------------------------------------------
# 4. Save all results
# ------------------------------------------------------------

fold_ablation_df.to_csv(
    OUTPUT_DIR / "cwru_48k_ablation_fold_metrics.csv",
    index=False
)

ablation_predictions_df.to_csv(
    OUTPUT_DIR / "cwru_48k_ablation_predictions.csv",
    index=False
)

ablation_summary_df.to_csv(
    OUTPUT_DIR / "cwru_48k_ablation_summary.csv",
    index=False
)

print("\nAblation evaluation saved successfully.")
print("Output directory:", OUTPUT_DIR)


In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

RESULTS_DIR = PROJECT_ROOT / "results" / "cwru_classification"

summary_path = RESULTS_DIR / "cwru_48k_ablation_summary.csv"
fold_path = RESULTS_DIR / "cwru_48k_ablation_fold_metrics.csv"

print("Results directory exists:", RESULTS_DIR.exists())
print("Summary file exists:", summary_path.exists())
print("Fold metrics file exists:", fold_path.exists())

assert summary_path.exists(), "Ablation summary CSV not found."
assert fold_path.exists(), "Fold metrics CSV not found."

ablation_summary = pd.read_csv(summary_path)
ablation_folds = pd.read_csv(fold_path)

print("\nSUMMARY FILE")
print("Shape:", ablation_summary.shape)
print("Columns:", ablation_summary.columns.tolist())

print("\nFOLD METRICS FILE")
print("Shape:", ablation_folds.shape)
print("Columns:", ablation_folds.columns.tolist())

print("\nFOLD METRICS PREVIEW")
display(ablation_folds.head())

In [ ]:
from pathlib import Path
import pandas as pd

# Locate the saved results directory
candidates = [
    Path.cwd() / "results" / "cwru_classification",
    Path.cwd().parent / "results" / "cwru_classification"
]

RESULTS_DIR = next(
    (p for p in candidates if p.exists()),
    None
)

assert RESULTS_DIR is not None, "CWRU results directory not found."

summary_path = RESULTS_DIR / "cwru_48k_ablation_summary.csv"
fold_path = RESULTS_DIR / "cwru_48k_ablation_fold_metrics.csv"

ablation_summary = pd.read_csv(summary_path)
ablation_folds = pd.read_csv(fold_path)

print("Summary shape:", ablation_summary.shape)
print("Fold metrics shape:", ablation_folds.shape)

print("\nSummary columns:")
print(ablation_summary.columns.tolist())

print("\nFold metrics columns:")
print(ablation_folds.columns.tolist())

In [ ]:
comparison = ablation_summary[
    [
        "feature_group",
        "feature_count",
        "mean_accuracy",
        "std_accuracy",
        "mean_macro_f1",
        "pooled_accuracy",
        "pooled_macro_f1"
    ]
].copy()

comparison["feature_reduction_pct"] = (
    (1 - comparison["feature_count"] / 37) * 100
).round(1)

comparison = comparison.sort_values(
    "feature_count"
).reset_index(drop=True)

display(comparison)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 6))

ax.errorbar(
    comparison["feature_count"],
    comparison["mean_accuracy"],
    yerr=comparison["std_accuracy"],
    fmt="o",
    capsize=5
)

for _, row in comparison.iterrows():
    ax.annotate(
        row["feature_group"],
        (row["feature_count"], row["mean_accuracy"]),
        xytext=(5, 7),
        textcoords="offset points",
        fontsize=8
    )

ax.set_xlabel("Number of Features")
ax.set_ylabel("Mean Accuracy Across Four Folds")
ax.set_title("CWRU Feature Ablation: Accuracy vs Feature Count")
ax.set_ylim(0.85, 1.02)
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Order configurations by number of features
feature_order = (
    ablation_folds[
        ["feature_group", "feature_count"]
    ]
    .drop_duplicates()
    .sort_values("feature_count")["feature_group"]
    .tolist()
)

# Arrange accuracy by feature group and held-out load
heatmap_data = (
    ablation_folds
    .pivot(
        index="feature_group",
        columns="held_out_load_hp",
        values="accuracy"
    )
    .reindex(feature_order)
    .sort_index(axis=1)
)

display(heatmap_data)

In [ ]:
fig, ax = plt.subplots(figsize=(9, 6))

matrix = heatmap_data.to_numpy()

im = ax.imshow(
    matrix,
    aspect="auto",
    vmin=0.80,
    vmax=1.00
)

ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(
    [f"{int(x)} HP" for x in heatmap_data.columns]
)

ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)

# Write accuracy values inside the cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(
            j, i,
            f"{matrix[i, j]:.2f}",
            ha="center",
            va="center"
        )

ax.set_title("CWRU Ablation: Accuracy Across Held-out Loads")
ax.set_xlabel("Held-out Motor Load")
ax.set_ylabel("Feature Configuration")

fig.colorbar(im, ax=ax, label="Accuracy")

plt.tight_layout()
plt.show()

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()) / "results" / "cwru_classification")
)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

comparison_path = (
    OUTPUT_DIR / "cwru_48k_ablation_comparison.csv"
)

comparison.to_csv(comparison_path, index=False)

print("Comparison table saved:")
print(comparison_path)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 6))

matrix = heatmap_data.to_numpy()

im = ax.imshow(
    matrix,
    aspect="auto",
    vmin=0.80,
    vmax=1.00
)

ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(
    [f"{int(x)} HP" for x in heatmap_data.columns]
)

ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)

for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax.text(
            j, i,
            f"{matrix[i, j]:.2f}",
            ha="center",
            va="center"
        )

ax.set_title("CWRU Ablation: Accuracy Across Held-out Loads")
ax.set_xlabel("Held-out Motor Load")
ax.set_ylabel("Feature Configuration")

fig.colorbar(im, ax=ax, label="Accuracy")

plt.tight_layout()

heatmap_path = (
    OUTPUT_DIR / "cwru_48k_ablation_heatmap.png"
)

fig.savefig(
    heatmap_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("Heatmap saved:", heatmap_path)

In [ ]:
from pathlib import Path
import joblib
import numpy as np
import pandas as pd

PROJECT_ROOT = Path(
    str((Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()))
)

DATA_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "cwru_48k_combined_features.joblib"
)

print("Dataset exists:", DATA_PATH.exists())
assert DATA_PATH.exists(), "Combined feature dataset not found."

cwru_data = joblib.load(DATA_PATH)

# Load independently of previous notebook variables
X_audit = np.asarray(cwru_data["features"])
meta_audit = cwru_data["metadata"].copy()
feature_names_audit = list(cwru_data["feature_names"])

print("\nDATASET STRUCTURE")
print("Feature matrix:", X_audit.shape)
print("Metadata:", meta_audit.shape)
print("Number of features:", len(feature_names_audit))
print("All values finite:", np.isfinite(X_audit).all())

print("\nFEATURE NAMES")
print(feature_names_audit)

print("\nMETADATA COLUMNS")
print(meta_audit.columns.tolist())

print("\nCLASS DISTRIBUTION")
print(meta_audit["fault_class"].value_counts())

print("\nCLASS VS MOTOR LOAD")
display(pd.crosstab(
    meta_audit["fault_class"],
    meta_audit["motor_load_hp"]
))

print("\nRECORDING COUNTS")
print("Unique recordings:", meta_audit["filename"].nunique())

print(
    meta_audit.groupby("fault_class")["filename"]
    .nunique()
)

# Exact duplicate feature-vector audit
feature_df_audit = pd.DataFrame(
    X_audit,
    columns=feature_names_audit
)

print(
    "\nExact duplicate feature rows:",
    feature_df_audit.duplicated().sum()
)

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

import pandas as pd
import numpy as np

# Use all 37 features
X = X_audit.copy()

# Encode fault labels consistently
encoder = LabelEncoder()
y = encoder.fit_transform(meta_audit["fault_class"].astype(str))

# Motor load determines the held-out fold
groups = meta_audit["motor_load_hp"].to_numpy()

logo = LeaveOneGroupOut()
audit_results = []

for fold, (train_idx, test_idx) in enumerate(
    logo.split(X, y, groups), start=1
):
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(
            class_weight="balanced",
            max_iter=3000,
            solver="lbfgs",
            random_state=42
        ))
    ])

    model.fit(X[train_idx], y[train_idx])

    train_pred = model.predict(X[train_idx])
    test_pred = model.predict(X[test_idx])

    train_accuracy = accuracy_score(
        y[train_idx], train_pred
    )

    test_accuracy = accuracy_score(
        y[test_idx], test_pred
    )

    audit_results.append({
        "held_out_load_hp": int(groups[test_idx][0]),
        "train_windows": len(train_idx),
        "test_windows": len(test_idx),
        "train_accuracy": train_accuracy,
        "test_accuracy": test_accuracy,
        "accuracy_gap": train_accuracy - test_accuracy,
        "train_balanced_accuracy": balanced_accuracy_score(
            y[train_idx], train_pred
        ),
        "test_balanced_accuracy": balanced_accuracy_score(
            y[test_idx], test_pred
        ),
        "train_macro_f1": f1_score(
            y[train_idx], train_pred, average="macro"
        ),
        "test_macro_f1": f1_score(
            y[test_idx], test_pred, average="macro"
        )
    })

training_test_audit = pd.DataFrame(audit_results)

display(training_test_audit.round(4))

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import LabelEncoder
import pandas as pd

recording_groups = (
    meta_audit["filename"].astype(str).to_numpy()
)

group_cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

recording_splits = list(
    group_cv.split(X_audit, y, recording_groups)
)

for fold, (train_idx, test_idx) in enumerate(
    recording_splits, start=1
):
    train_files = set(recording_groups[train_idx])
    test_files = set(recording_groups[test_idx])

    overlap = train_files.intersection(test_files)

    train_classes = encoder.inverse_transform(
        y[train_idx]
    )

    test_classes = encoder.inverse_transform(
        y[test_idx]
    )

    print(f"\nFOLD {fold}")
    print("Training recordings:", len(train_files))
    print("Testing recordings:", len(test_files))
    print("Recording overlap:", len(overlap))

    print("\nTraining class distribution:")
    print(pd.Series(train_classes).value_counts())

    print("\nTesting class distribution:")
    print(pd.Series(test_classes).value_counts())

    assert len(overlap) == 0

In [ ]:
print("Total features:", len(feature_names_audit))

for i, name in enumerate(feature_names_audit):
    print(i, name)

In [ ]:
# Define feature groups using the verified feature names

TIME_FEATURES_AUDIT = feature_names_audit[0:9]

FREQUENCY_FEATURES_AUDIT = feature_names_audit[9:22]

ENVELOPE_FEATURES_AUDIT = feature_names_audit[22:37]

FEATURE_CONFIGS_AUDIT = {
    "Frequency-only": FREQUENCY_FEATURES_AUDIT,
    "Envelope-only": ENVELOPE_FEATURES_AUDIT,
    "All 37 Features": feature_names_audit
}

# Verify the configurations
for group, features in FEATURE_CONFIGS_AUDIT.items():
    missing = set(features) - set(feature_names_audit)

    print(f"{group}: {len(features)} features")
    assert len(missing) == 0, f"Missing features: {missing}"

assert len(TIME_FEATURES_AUDIT) == 9
assert len(FREQUENCY_FEATURES_AUDIT) == 13
assert len(ENVELOPE_FEATURES_AUDIT) == 15
assert len(feature_names_audit) == 37

print("\nAll feature configurations verified.")

In [ ]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score
)

import numpy as np
import pandas as pd

# Rebuild labels and recording groups
y_grouped = LabelEncoder().fit_transform(
    meta_audit["fault_class"].astype(str)
)

groups_grouped = meta_audit["filename"].astype(str).to_numpy()

group_cv = StratifiedGroupKFold(
    n_splits=4,
    shuffle=True,
    random_state=42
)

grouped_splits = list(
    group_cv.split(X_audit, y_grouped, groups_grouped)
)

grouped_results = []

# Evaluate each feature configuration
for config_name, selected_features in FEATURE_CONFIGS_AUDIT.items():

    feature_indices = [
        feature_names_audit.index(name)
        for name in selected_features
    ]

    X_config = X_audit[:, feature_indices]

    print(f"\nEvaluating: {config_name}")

    for fold, (train_idx, test_idx) in enumerate(
        grouped_splits, start=1
    ):

        model = Pipeline([
            ("scaler", StandardScaler()),
            ("classifier", LogisticRegression(
                class_weight="balanced",
                max_iter=3000,
                solver="lbfgs",
                random_state=42
            ))
        ])

        model.fit(
            X_config[train_idx],
            y_grouped[train_idx]
        )

        train_pred = model.predict(X_config[train_idx])
        test_pred = model.predict(X_config[test_idx])

        train_acc = accuracy_score(
            y_grouped[train_idx], train_pred
        )
        test_acc = accuracy_score(
            y_grouped[test_idx], test_pred
        )

        train_bal = balanced_accuracy_score(
            y_grouped[train_idx], train_pred
        )
        test_bal = balanced_accuracy_score(
            y_grouped[test_idx], test_pred
        )

        train_f1 = f1_score(
            y_grouped[train_idx], train_pred,
            average="macro", zero_division=0
        )
        test_f1 = f1_score(
            y_grouped[test_idx], test_pred,
            average="macro", zero_division=0
        )

        grouped_results.append({
            "feature_group": config_name,
            "feature_count": len(selected_features),
            "fold": fold,
            "train_recordings": len(set(groups_grouped[train_idx])),
            "test_recordings": len(set(groups_grouped[test_idx])),
            "train_accuracy": train_acc,
            "test_accuracy": test_acc,
            "accuracy_gap": train_acc - test_acc,
            "train_balanced_accuracy": train_bal,
            "test_balanced_accuracy": test_bal,
            "balanced_accuracy_gap": train_bal - test_bal,
            "train_macro_f1": train_f1,
            "test_macro_f1": test_f1,
            "macro_f1_gap": train_f1 - test_f1
        })

        print(
            f"Fold {fold} | "
            f"Train: {train_acc:.3f} | "
            f"Test: {test_acc:.3f} | "
            f"Gap: {train_acc - test_acc:.3f}"
        )

grouped_results_df = pd.DataFrame(grouped_results)

print("\nGrouped evaluation completed.")
print("Models evaluated:", len(grouped_results_df))

display(grouped_results_df.round(4))

In [ ]:
from pathlib import Path

# Derive a condition key by removing the final load suffix
meta_audit["condition_key"] = (
    meta_audit["filename"]
    .astype(str)
    .apply(lambda x: Path(x).stem.rsplit("_", 1)[0])
)

condition_table = (
    meta_audit[
        ["filename", "fault_class", "motor_load_hp", "condition_key"]
    ]
    .drop_duplicates()
    .sort_values(["fault_class", "condition_key", "motor_load_hp"])
)

display(condition_table)

print("\nCondition identities per fault class:")

display(
    meta_audit.groupby("fault_class")["condition_key"]
    .nunique()
)